# Library dan Konfigurasi

## Import Library


In [1]:
import ast
import io
import json
import os
import re
import shutil
import subprocess
import sys
import stat
import time
import tokenize
from collections import defaultdict
from datetime import datetime
import random
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

# Formatter untuk standarisasi kode Python
import autopep8
import black

# Engine export Excel untuk pandas
import openpyxl

# Progress bar & notebook display
from tqdm.notebook import tqdm
from IPython.display import display, HTML

# Waktu
run_time = datetime.now()

# Konfigurasi visualisasi default
%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)

print("=" * 65)
print(f"{'LIBRARY INITIALIZATION':^65}")
print("-" * 65)
print(f"Python     : {sys.version.split()[0]}")
print(f"✅ {run_time.strftime('%Y-%m-%d %H:%M:%S')} - Libraries loaded successfully.")
print("-" * 65)

                     LIBRARY INITIALIZATION                      
-----------------------------------------------------------------
Python     : 3.10.6
✅ 2026-06-25 08:54:35 - Libraries loaded successfully.
-----------------------------------------------------------------


## Konfigurasi

In [2]:
# ==============================================================================
# DIRECTORY CONFIGURATION & INITIALIZATION
# Menentukan path utama, struktur folder dataset, dan file output
# ==============================================================================

# Root directory dan file input
BASE_DIR = r"D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code"
DATASET_FOLDER = "dataset(2)"
OUTPUT_FOLDER = "output(2)"

INPUT_GITHUB = os.path.join(BASE_DIR, "asli", "nim_github.txt")

# Struktur folder pipeline
DIRS = {
    "DATASET"      : os.path.join(BASE_DIR, DATASET_FOLDER),

    # Preprocessing
    "RAW"          : os.path.join(BASE_DIR, DATASET_FOLDER, "00_Raw"),
    "ANON"          : os.path.join(BASE_DIR, DATASET_FOLDER, "01_Raw_Anon"),
    "NORM"         : os.path.join(BASE_DIR, DATASET_FOLDER, "02_Normalized"),
    "CONV"         : os.path.join(BASE_DIR, DATASET_FOLDER, "03_Converted"),
    "CLEAN"        : os.path.join(BASE_DIR, DATASET_FOLDER, "04_Cleaned"),

    # Formatter experiment
    "AUTOPEP8"     : os.path.join(BASE_DIR, DATASET_FOLDER, "05a_Autopep8"),
    "BLACK"        : os.path.join(BASE_DIR, DATASET_FOLDER, "05b_Black"),
    
    "FILTERED"     : os.path.join(BASE_DIR, DATASET_FOLDER, "06_Filtered"),
    "FILTERED_PRAK"     : os.path.join(BASE_DIR, DATASET_FOLDER, "06_Filtered_prak"),
    "FILTERED_TGS"     : os.path.join(BASE_DIR, DATASET_FOLDER, "06_Filtered_tgs"),

    # AST & Graph
    "AST_P"          : os.path.join(BASE_DIR, DATASET_FOLDER, "07_AST_prak"),
    "AST_T"          : os.path.join(BASE_DIR, DATASET_FOLDER, "07_AST_tgs"),
    "AST_VISUAL_P"   : os.path.join(BASE_DIR, DATASET_FOLDER, "07a_AST_visual_prak"),
    "AST_VISUAL_T"   : os.path.join(BASE_DIR, DATASET_FOLDER, "07a_AST_visual_tgs"),
    "GRAPH_P"        : os.path.join(BASE_DIR, DATASET_FOLDER, "08_Graph_prak"),
    "GRAPH_T"        : os.path.join(BASE_DIR, DATASET_FOLDER, "08_Graph_tgs"),
    "INPUT_GRAPH_P"  : os.path.join(BASE_DIR, DATASET_FOLDER, "09_Graph2vec_Input_prak"),
    "INPUT_GRAPH_T"  : os.path.join(BASE_DIR, DATASET_FOLDER, "09_Graph2vec_Input_tgs"),
    "EMBEDDING_P"    : os.path.join(BASE_DIR, DATASET_FOLDER, "10_Graph2vec_Embedding_prak"),
    "EMBEDDING_T"    : os.path.join(BASE_DIR, DATASET_FOLDER, "10_Graph2vec_Embedding_tgs"),

    # Output
    "LOGS"         : os.path.join(BASE_DIR, OUTPUT_FOLDER),
    "RUNTIME_A"    : os.path.join(BASE_DIR, OUTPUT_FOLDER, "run_autopep8"),
    "RUNTIME_B"    : os.path.join(BASE_DIR, OUTPUT_FOLDER, "run_black"),
    "EVAL"         : os.path.join(BASE_DIR, OUTPUT_FOLDER, "eval"),
    "EVAL_P"       : os.path.join(BASE_DIR, OUTPUT_FOLDER, "eval", "praktikum"),
    "EVAL_T"       : os.path.join(BASE_DIR, OUTPUT_FOLDER, "eval", "tugas"),
    
    # folder output untuk grafik similarity
    "SIMILARITY"   : os.path.join(BASE_DIR, OUTPUT_FOLDER,"similarity"),
}

# File output penelitian
RESULTS = {
    # Preprocessing
    "CLONE_REPORT"    : os.path.join(DIRS["LOGS"], "01_clone_report.xlsx"),
    "NORM_REPORT"     : os.path.join(DIRS["LOGS"], "02_normalization_report.xlsx"),
    "CONV_REPORT"     : os.path.join(DIRS["LOGS"], "03_conversion_report.xlsx"),
    "CLEAN_REPORT"    : os.path.join(DIRS["LOGS"], "04_cleaning_report.xlsx"),

    # Formatter
    "ERR_AUTOPEP"     : os.path.join(DIRS["LOGS"], "05a_autopep_errors.json"),
    "ERR_BLACK"       : os.path.join(DIRS["LOGS"], "05b_black_errors.json"),

    # Runtime Autopep8
    "RUN_PROJECT_A"   : os.path.join(DIRS["RUNTIME_A"], "a_runtime_project_AUTOPEP8.xlsx"),
    "RUN_FUNCTION_A"  : os.path.join(DIRS["RUNTIME_A"], "b_runtime_function_AUTOPEP8.xlsx"),
    "RUN_COMPARE_A"   : os.path.join(DIRS["RUNTIME_A"], "c_runtime_compare_AUTOPEP8.xlsx"),

    # Runtime Black
    "RUN_PROJECT_B"   : os.path.join(DIRS["RUNTIME_B"], "a_runtime_project_BLACK.xlsx"),
    "RUN_FUNCTION_B"  : os.path.join(DIRS["RUNTIME_B"], "b_runtime_function_BLACK.xlsx"),
    "RUN_COMPARE_B"   : os.path.join(DIRS["RUNTIME_B"], "c_runtime_compare_BLACK.xlsx"),

    # Submission
    "SUBMISSION"      : os.path.join(DIRS["LOGS"], "06_submission_report.xlsx"),

    # AST & Graph
    "EXTRACT_AST_P"     : os.path.join(DIRS["LOGS"], "07_AST_report_prak.xlsx"),
    "EXTRACT_AST_T"     : os.path.join(DIRS["LOGS"], "07_AST_report_tgs.xlsx"),
    "CONSTRUCT_GRAPH_P" : os.path.join(DIRS["LOGS"], "08_Graph_report_prak.xlsx"),
    "CONSTRUCT_GRAPH_T" : os.path.join(DIRS["LOGS"], "08_Graph_report_tgs.xlsx"),
    "LIST_GRAPH_P"      : os.path.join(DIRS["LOGS"], "09_list_Graph_report_prak.xlsx"),
    "LIST_GRAPH_T"      : os.path.join(DIRS["LOGS"], "09_list_Graph_report_tgs.xlsx"),

    # Graph2Vec
    "EMBEDDING_REPORT_P": os.path.join(DIRS["LOGS"], "10_embedding_report_prak.xlsx"),
    "EMBEDDING_REPORT_T": os.path.join(DIRS["LOGS"], "10_embedding_report_tgs.xlsx"),
    "EMBEDDING_VECTOR_P": os.path.join(DIRS["LOGS"], "10a_embedding_vector_prak.xlsx"),
    "EMBEDDING_VECTOR_T": os.path.join(DIRS["LOGS"], "10a_embedding_vector_tgs.xlsx"),

    # Similarity PRAKTIKUM
    "SIMILARITY_P"      : os.path.join(DIRS["LOGS"], "11_similarity_report_prak.xlsx"),
    "SIMILARITY_MODUL_P" : os.path.join(DIRS['LOGS'], "11a_similarity_per_Modul_prak.xlsx"),
    "SIMILARITY_SUMMARY_P" : os.path.join(DIRS['LOGS'], "11b_similarity_summary_prak.xlsx"),
    
    # Euclidean Similarity
    "EUCLIDEAN_P" : os.path.join(DIRS['LOGS'], "12_euclidean_similarity_prak.xlsx"),
    "EUCLIDEAN_MODUL_P" : os.path.join(DIRS['LOGS'], "12a_euclidean_similarity_modul_prak.xlsx"),
    
    "SIMILARITY_COMPARE_P" : os.path.join(DIRS['LOGS'], "13_similarity_comparison_prak.xlsx"),
    
    # Similarity TUGAS
    "SIMILARITY_T"      : os.path.join(DIRS["LOGS"], "11_similarity_report_tgs.xlsx"),
    "SIMILARITY_MODUL_T" : os.path.join(DIRS['LOGS'], "11a_similarity_per_Modul_tgs.xlsx"),
    "SIMILARITY_SUMMARY_T" : os.path.join(DIRS['LOGS'], "11b_similarity_summary_tgs.xlsx"),
    
    # Euclidean Similarity
    "EUCLIDEAN_T" : os.path.join(DIRS['LOGS'], "12_euclidean_similarity_tgs.xlsx"),
    "EUCLIDEAN_MODUL_T" : os.path.join(DIRS['LOGS'], "12a_euclidean_similarity_modul_tgs.xlsx"),
    
    "SIMILARITY_COMPARE_T" : os.path.join(DIRS['LOGS'], "13_similarity_comparison_tgs.xlsx"),
    
    # METRICS
    "METRICS"         : os.path.join(DIRS["LOGS"], "metrics_evaluation_report.xlsx"),
    "EXECUTION_TIME"  : os.path.join(DIRS["LOGS"], "execution_time_report.xlsx"),
}

# VALIDATION & DIRECTORY INITIALIZATION
print("=" * 70)
print(f"{'DIRECTORY CONFIGURATION':^70}")
print("=" * 70)

# Validasi root directory
if not os.path.isdir(BASE_DIR):
    raise FileNotFoundError(f"❌ BASE_DIR tidak ditemukan: {BASE_DIR}")

# Validasi file input
if not os.path.isfile(INPUT_GITHUB):
    raise FileNotFoundError(f"❌ File input tidak ditemukan: {INPUT_GITHUB}")

if not INPUT_GITHUB.endswith(".txt"):
    raise ValueError("❌ File input harus berekstensi .txt")

if os.path.getsize(INPUT_GITHUB) == 0:
    raise ValueError(f"❌ File input kosong: {INPUT_GITHUB}")

# Membuat folder pipeline
folders_created = 0

for name, path in DIRS.items():
    if not os.path.exists(path):
        os.makedirs(path, exist_ok=True)
        folders_created += 1
        status = "[NEW]"
    elif not os.path.isdir(path):
        raise NotADirectoryError(f"❌ Path bukan folder: {path}")
    else:
        status = "[EXISTS]"
    print(f"{status:<10} {name:<12} : {path}")

# Validasi parent folder output
missing_results_parent = []

for name, path in RESULTS.items():
    parent_dir = os.path.dirname(path)
    if not os.path.exists(parent_dir):
        missing_results_parent.append(
            f"{name} : {parent_dir}"
        )

if missing_results_parent:
    raise FileNotFoundError(
        "❌ Parent folder RESULTS tidak ditemukan:\n"
        + "\n".join(missing_results_parent)
    )

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("-" * 70)
print(f"Folders Created  : {folders_created}")
print(f"Dataset Root     : {DIRS['DATASET']}")
print(f"Input File       : {INPUT_GITHUB}")
print("=" * 70)
print(f"Diproses pada {timestamp}")

                       DIRECTORY CONFIGURATION                        
[EXISTS]   DATASET      : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)
[EXISTS]   RAW          : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\00_Raw
[EXISTS]   ANON         : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\01_Raw_Anon
[EXISTS]   NORM         : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\02_Normalized
[EXISTS]   CONV         : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\03_Converted
[EXISTS]   CLEAN        : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\04_Cleaned
[EXISTS]   AUTOPEP8     : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\05a_Autopep8
[EXISTS]   BLACK        : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\05b_Black
[EXISTS]   FILTERED     : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\06_Filtered
[EXISTS]   FILTERED_PRAK : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\06_Filtered_prak
[EXISTS]   FILTERED_TGS : D:\PUTRI\D4\S

In [3]:
# ==============================================================================
# EVALUATION CONFIG DIR
# ==============================================================================
DATASET_EVAL_FOLDER = DATASET_FOLDER + "_eval"
DATASET_EVAL_DIR = os.path.join(BASE_DIR, DATASET_EVAL_FOLDER)
DATASET_EVAL_P = os.path.join(DATASET_EVAL_DIR, "praktikum")
DATASET_EVAL_T = os.path.join(DATASET_EVAL_DIR, "tugas")

EVAL_P = {
    # PRAKTIKUM
    "SAMPLE"       : os.path.join(DATASET_EVAL_P, "06_SAMPLE"),
    "AST"          : os.path.join(DATASET_EVAL_P, "07_AST"),
    "AST_VISUAL"   : os.path.join(DATASET_EVAL_P, "07a_AST_visual"),
    "GRAPH"        : os.path.join(DATASET_EVAL_P, "08_Graph"),
    "INPUT_GRAPH"  : os.path.join(DATASET_EVAL_P, "09_Graph2vec_Input"),
    "EMBEDDING"    : os.path.join(DATASET_EVAL_P, "10_Graph2vec_Embedding"),
}

EVAL_T = {
    # TUGAS
    "SAMPLE"       : os.path.join(DATASET_EVAL_T, "06_SAMPLE"),
    "AST"          : os.path.join(DATASET_EVAL_T, "07_AST"),
    "AST_VISUAL"   : os.path.join(DATASET_EVAL_T, "07a_AST_visual"),
    "GRAPH"        : os.path.join(DATASET_EVAL_T, "08_Graph"),
    "INPUT_GRAPH"  : os.path.join(DATASET_EVAL_T, "09_Graph2vec_Input"),
    "EMBEDDING"    : os.path.join(DATASET_EVAL_T, "10_Graph2vec_Embedding"),
}

RESULTS_EVAL_P = {
    "SAMPLING_REPORT" : os.path.join(DIRS["EVAL_P"], "06_sampling_report.xlsx"),
    # AST & Graph
    "EXTRACT_AST"     : os.path.join(DIRS["EVAL_P"], "07_AST_report.xlsx"),
    "CONSTRUCT_GRAPH" : os.path.join(DIRS["EVAL_P"], "08_Graph_report.xlsx"),
    "LIST_GRAPH"      : os.path.join(DIRS["EVAL_P"], "09_list_Graph_report.xlsx"),

    # Graph2Vec
    "EMBEDDING_REPORT": os.path.join(DIRS["EVAL_P"], "10_embedding_report.xlsx"),
    "EMBEDDING_VECTOR": os.path.join(DIRS["EVAL_P"], "10a_embedding_vector.xlsx"),

    # Similarity
    "SIMILARITY"      : os.path.join(DIRS["EVAL_P"], "11_similarity_report.xlsx"),
    "SIMILARITY_MODUL" : os.path.join(DIRS['EVAL_P'], "11a_similarity_per_Modul.xlsx"),
    "SIMILARITY_SUMMARY" : os.path.join(DIRS['EVAL_P'], "11b_similarity_summary.xlsx"),
    
    # Euclidean Similarity
    "EUCLIDEAN" : os.path.join(DIRS['EVAL_P'], "12_euclidean_similarity.xlsx"),
    "EUCLIDEAN_MODUL" : os.path.join(DIRS['EVAL_P'], "12a_euclidean_similarity_modul.xlsx"),
    
    "SIMILARITY_COMPARE" : os.path.join(DIRS['EVAL_P'], "13_similarity_comparison.xlsx"),
}

RESULTS_EVAL_T = {
    "SAMPLING_REPORT" : os.path.join(DIRS["EVAL_T"], "06_sampling_report.xlsx"),
    # AST & Graph
    "EXTRACT_AST"     : os.path.join(DIRS["EVAL_T"], "07_AST_report.xlsx"),
    "CONSTRUCT_GRAPH" : os.path.join(DIRS["EVAL_T"], "08_Graph_report.xlsx"),
    "LIST_GRAPH"      : os.path.join(DIRS["EVAL_T"], "09_list_Graph_report.xlsx"),

    # Graph2Vec
    "EMBEDDING_REPORT": os.path.join(DIRS["EVAL_T"], "10_embedding_report.xlsx"),
    "EMBEDDING_VECTOR": os.path.join(DIRS["EVAL_T"], "10a_embedding_vector.xlsx"),

    # Similarity
    "SIMILARITY"      : os.path.join(DIRS["EVAL_T"], "11_similarity_report.xlsx"),
    "SIMILARITY_MODUL" : os.path.join(DIRS['EVAL_T'], "11a_similarity_per_Modul.xlsx"),
    "SIMILARITY_SUMMARY" : os.path.join(DIRS['EVAL_T'], "11b_similarity_summary.xlsx"),
    
    # Euclidean Similarity
    "EUCLIDEAN" : os.path.join(DIRS['EVAL_T'], "12_euclidean_similarity.xlsx"),
    "EUCLIDEAN_MODUL" : os.path.join(DIRS['EVAL_T'], "12a_euclidean_similarity_modul.xlsx"),
    
    "SIMILARITY_COMPARE" : os.path.join(DIRS['EVAL_T'], "13_similarity_comparison.xlsx"),
}


# INITIALIZATION & VALIDATION EVALUATION DIRECTORY
folders_created = 0
print(f"{'='*20} EVALUATION DIRECTORY CONFIGURATION {'='*20}")

# Validasi dan pembuatan folder evaluasi
for cfg_name, config in [("EVAL_P", EVAL_P), ("EVAL_T", EVAL_T)]:
    print(f"\n[{cfg_name}]")
    for name, path in config.items():
        if not os.path.exists(path):
            os.makedirs(path, exist_ok=True)
            folders_created += 1
            status = "[NEW]"
        elif not os.path.isdir(path):
            raise NotADirectoryError(f"❌ Path bukan folder: {path}")
        else:
            status = "[EXISTS]"
        print(f"{status:<10} {name:<15} : {path}")

# ------------------------------------------------------------------------------
# VALIDASI PARENT DIRECTORY UNTUK FILE REPORT
# ------------------------------------------------------------------------------
missing_results = []
for grp_name, group in [("RESULTS_EVAL_P", RESULTS_EVAL_P), ("RESULTS_EVAL_T", RESULTS_EVAL_T)]:
    for name, path in group.items():
        parent_dir = os.path.dirname(path)
        if not os.path.exists(parent_dir):
            missing_results.append(f"{grp_name}.{name} -> {parent_dir}")

if missing_results:
    raise FileNotFoundError("❌ Parent folder RESULTS tidak ditemukan:\n" + "\n".join(missing_results))

# ------------------------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------------------------
print(f"\n{'-'*70}")
print(f"Folders Created     : {folders_created}")
print(f"Evaluation Root     : {DATASET_EVAL_DIR}")
print(f"Praktikum Root      : {DATASET_EVAL_P}")
print(f"Tugas Root          : {DATASET_EVAL_T}")
print("=" * 70)
print(f"Diproses pada {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

==================== EVALUATION DIRECTORY CONFIGURATION ====================

[EVAL_P]
[EXISTS]   SAMPLE          : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\praktikum\06_SAMPLE
[EXISTS]   AST             : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\praktikum\07_AST
[EXISTS]   AST_VISUAL      : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\praktikum\07a_AST_visual
[EXISTS]   GRAPH           : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\praktikum\08_Graph
[EXISTS]   INPUT_GRAPH     : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\praktikum\09_Graph2vec_Input
[EXISTS]   EMBEDDING       : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\praktikum\10_Graph2vec_Embedding

[EVAL_T]
[EXISTS]   SAMPLE          : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\tugas\06_SAMPLE
[EXISTS]   AST             : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\tugas\07_AST
[EXISTS]   AST_VISUAL      : D:\PUTRI\D4\SE

## Helper Function

### helper count

In [4]:
# HELPER FUNCTION count

def count_all_files(dataset_path, extensions=None, exclude_dirs=None):
    total_files = 0
    by_extension = {}

    exclude_dirs = set(exclude_dirs or [])

    for root, dirs, files in os.walk(dataset_path):
        # skip folder tertentu
        dirs[:] = [d for d in dirs if d not in exclude_dirs]

        for file in files:
            ext = os.path.splitext(file)[1].lower()

            # filter ekstensi jika diberikan
            if extensions and ext not in extensions:
                continue

            total_files += 1
            by_extension[ext] = by_extension.get(ext, 0) + 1

    return {
        "total_files": total_files,
        "by_extension": by_extension
    }

def count_students(data, nim_column="nim"):
    """
    Menghitung jumlah mahasiswa dari berbagai tipe data.
    """

    # Folder dataset
    if isinstance(data, str):
        return sum(
            1
            for item in os.listdir(data)
            if os.path.isdir(os.path.join(data, item))
        )

    # DataFrame
    if isinstance(data, pd.DataFrame):
        return data[nim_column].nunique()

    # List metadata
    if isinstance(data, list):
        return len({
            row[nim_column]
            for row in data
            if nim_column in row
        })

    raise TypeError(
        "data harus berupa path folder, DataFrame, atau list metadata"
    )
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"✅ Function dijalankan pada {timestamp}")

✅ Function dijalankan pada 2026-06-25 08:54:35


In [5]:
def extract_assignment_name(filename):
    """
    NIM_MODUL_FILE.py
    -> FILE
    """
    filename = os.path.splitext(
        os.path.basename(filename)
    )[0]
    parts = filename.split("_")
    if len(parts) < 3:
        return None, None, filename
    nim = parts[0]
    modul = parts[1]
    nama_file = "_".join(parts[2:])
    return nim, modul, nama_file

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"✅ Function dijalankan pada {timestamp}")

✅ Function dijalankan pada 2026-06-25 08:54:35


In [6]:
def get_path_size(path, unit="kb"):
    """Menghitung kapasitas memori file/folder secara rekursif."""
    if not os.path.exists(path):
        return 0

    total_size = 0
    if os.path.isfile(path):
        try: total_size = os.path.getsize(path)
        except Exception: return 0
    else:
        for root, _, files in os.walk(path):
            for file in files:
                file_path = os.path.join(root, file)
                try:
                    if os.path.exists(file_path): total_size += os.path.getsize(file_path)
                except Exception: pass

    unit = unit.lower()
    factors = {"byte": 1, "kb": 1024, "mb": 1024**2, "gb": 1024**3}
    if unit in factors:
        return total_size if unit == "byte" else round(total_size / factors[unit], 2)
    else:
        raise ValueError("Unit tidak valid. Gunakan: byte/kb/mb/gb")


def count_loc(file_path):
    """Menghitung kuantitas baris kode aktif (tanpa baris kosong)."""
    if not os.path.exists(file_path):
        return 0

    ext = os.path.splitext(file_path)[1].lower()
    try:
        if ext == ".py":
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                return sum(1 for line in f if line.strip())
        elif ext == ".ipynb":
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                notebook = json.load(f)
            loc = 0
            for cell in notebook.get("cells", []):
                if cell.get("cell_type") == "code":
                    source = cell.get("source", [])
                    loc += sum(1 for line in source if str(line).strip())
            return loc
        return 0
    except Exception:
        return 0

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"✅ Function dijalankan pada {timestamp}")

✅ Function dijalankan pada 2026-06-25 08:54:35


### helper timer

In [7]:
# ==============================================================================
# RUNTIME METRICS
# ==============================================================================
from openpyxl.styles import Alignment, Font
import re
def start_timer():
    """Memulai timer komputasi."""
    return time.perf_counter()

def stage_sort_key(stage):
    """
    Mengurutkan stage seperti:
    1_stage
    2_stage
    ...
    10_stage
    10a_stage
    10b_stage
    11_stage
    """
    
    match = re.match(r"(\d+)([a-zA-Z]*)", str(stage))

    if match:
        number = int(match.group(1))
        suffix = match.group(2).lower()
        return (number, suffix)

    return (9999, str(stage))

def save_execution_time(start_time, stage, total_mahasiswa, total_file, total_size_kb, output_file=RESULTS["EXECUTION_TIME"]):
    """
    Simpan atau update metrik runtime berdasarkan stage ke Excel.
    """
    execution_time = round(time.perf_counter() - start_time, 2)
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    new_data = {
        "stage": stage,
        "total_mahasiswa": total_mahasiswa,
        "total_file": total_file,
        "waktu_eksekusi_s": execution_time,
        "total_size_kb": round(total_size_kb, 2),
        "timestamp": timestamp
    }

    # Logika Update/Insert
    if os.path.exists(output_file):
        df = pd.read_excel(output_file)
        if not df.empty and stage in df["stage"].values:
            df.loc[
                df["stage"] == stage,
                [
                    "total_mahasiswa",
                    "total_file",
                    "waktu_eksekusi_s",
                    "total_size_kb",
                    "timestamp"
                ]
            ] = [
                total_mahasiswa,
                total_file,
                execution_time,
                round(total_size_kb, 2),
                timestamp
            ]
        else:
            df = pd.concat([df, pd.DataFrame([new_data])], ignore_index=True)
    else:
        df = pd.DataFrame([new_data])

    df = (
        df.assign(
            _sort_key=df["stage"].apply(stage_sort_key)
        )
        .sort_values("_sort_key")
        .drop(columns="_sort_key")
        .reset_index(drop=True)
    )
    
    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        df.to_excel(writer, index=False, sheet_name="Execution Time")

        ws = writer.sheets["Execution Time"]

        ws.freeze_panes = "A2"
        # Header bold + center
        for cell in ws[1]:
            cell.font = Font(bold=True)
            cell.alignment = Alignment(
                horizontal="center",
                vertical="center"
            )
        for col in ws.columns:
            max_len = max(
                (
                    len(str(cell.value))
                    if cell.value is not None
                    else 0
                )
                for cell in col
            )
            ws.column_dimensions[
                col[0].column_letter
            ].width = min(max_len + 4, 60)

        for row in ws.iter_rows(min_row=2):
            for cell in row:
                cell.alignment = Alignment(vertical="center")
            
    print(f"\n[SELESAI] {stage} | {execution_time} s | {total_file} file")
    return execution_time

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"✅ Runtime Metrics Logger siap digunakan pada {timestamp}")

✅ Runtime Metrics Logger siap digunakan pada 2026-06-25 08:54:35


# Preprocessing

## 1. Pengumpulan Data (Cloning)


In [20]:
# CLONING GITHUB REPOSITORIES
# Kumpulan fungsi bantu untuk proses cloning repository
# ------------------------------------------------------------------------------
# token github
from dotenv import load_dotenv
load_dotenv()  # Memuat variabel lingkungan dari file .env jika ada
TOKEN_GIT = os.getenv("GITHUB_TOKEN", "")  # Ambil token dari .env atau gunakan default
print(f"✅ Token GitHub {'ditemukan' if TOKEN_GIT else 'tidak ditemukan, menggunakan URL tanpa autentikasi'}.")

# Fungsi untuk menyisipkan token GitHub ke dalam URL untuk autentikasi
def add_token(url, token: str = None):
    # Validasi URL
    if not isinstance(url, str) or not url.strip():
        raise ValueError("URL repository tidak valid.")
    # Jika token kosong
    if token is None or str(token).strip() == "":
        return url
    # Hanya proses URL GitHub HTTPS
    if url.startswith("https://github.com"):
        return url.replace(
            "https://",
            f"https://{token}@"
        )
    return url

# Fungsi untuk membaca file dataset
def load_dataset(file_path):
    dataset = []
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"     [WARNING] File dataset tidak ditemukan: {file_path}")      
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            clean_line = line.strip()
            if not clean_line or "|" not in clean_line:
                continue
            parts = clean_line.split("|")
            nim = parts[0].strip()
            url = parts[1].strip()
            dataset.append((nim, url))
    return dataset

# Handler untuk menghapus file/folder yang bersifat read-only
def remove_readonly(func, path, exc_info):
    try:
        os.chmod(path, stat.S_IWRITE)
        func(path)
    except Exception as e:
        print(f"    [WARNING] Gagal hapus : {path} \n       -> {str(e)}")

# Fungsi untuk menghapus semua file kecuali .py dan .ipynb, serta membersihkan folder kosong
def keep_only_code_files(repo_path):
    valid_extensions = [".py", ".ipynb"]
    ignored_dirs = [".git", "__pycache__", ".idea", ".vscode", "venv", ".venv", "env", "build", "dist", "node_modules", "colab lama", ".ipynb_checkpoints"]

    # HAPUS FILE NON-CODE
    for root, dirs, files in os.walk(repo_path, topdown=True):
        # Hapus ignored directories
        for d in dirs[:]:
            if d in ignored_dirs:
                dir_path = os.path.join(root, d)
                try:
                    shutil.rmtree(dir_path,onerror=remove_readonly)
                    dirs.remove(d)

                except Exception as e:
                    print(
                        f"[WARNING] Gagal hapus folder:\n"
                        f"{dir_path}\n"
                        f"-> {str(e)}"
                    )

        for file in files:
            file_path = os.path.join(root, file)
            _, ext = os.path.splitext(file)

            # Hanya simpan .py dan .ipynb
            if ext.lower() not in valid_extensions:
                try:
                    os.remove(file_path)
                except Exception as e:
                    print(
                        f"[WARNING] Gagal menghapus file:\n"
                        f"{file_path}\n"
                        f"-> {str(e)}"
                    )

    # Hapus folder yang menjadi kosong setelah pembersihan file (secara bottom-up)
    for root, dirs, files in os.walk(repo_path, topdown=False):
        try:
            # Jangan hapus root utama repository
            if root == repo_path:
                continue
            if not os.listdir(root):
                os.rmdir(root)
        except Exception:
            pass

# Fungsi untuk menghitung jumlah file .py dan .ipynb dalam sebuah direktori
def count_code_files(repo_path):
    py_count = sum(1 for root, _, files in os.walk(repo_path)
                for f in files if f.endswith(".py"))
    ipynb_count = sum(1 for root, _, files in os.walk(repo_path)
                    for f in files if f.endswith(".ipynb"))
    return py_count, ipynb_count

# Fungsi untuk menghitung jumlah folder (modul) di level utama repo
def count_modules(repo_path):
    try:
        return len([d for d in os.listdir(repo_path)
                    if os.path.isdir(os.path.join(repo_path, d)) and d != ".git"])
    except:
        return 0
    

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"✅ Diproses pada {timestamp}")

✅ Token GitHub ditemukan.
✅ Diproses pada 2026-06-10 02:03:07


In [21]:
# CLONING EXECUTION
# ------------------------------------------------------------------------------
results = []

def run_cloning_process(repo_list):
    global success_count, fail_count, skip_count
    global total_py_files, total_ipynb_files

    # Inisialisasi ulang counter
    success_count = fail_count = skip_count = 0
    total_py_files = total_ipynb_files = 0
    results.clear()

    total_start_time = start_timer()
    with tqdm(total=len(repo_list), desc="Cloning Repos", unit="repo") as pbar:
        for nim, url in repo_list:
            repo_start_time = time.time()
            target_path = os.path.join(DIRS['RAW'], nim)

            # Helper internal untuk menyimpan data hasil ke list
            def save_log(status, message, module=0, py=0, ipynb=0, repo_size=0, exec_time=0):
                results.append({
                    "nim": nim, "url": url, "status": status, "message": message,
                    "module": module, "py_files": py, "ipynb_files": ipynb, "total_files": py + ipynb, "repo_size_kb": repo_size, "execution_time_sec": round(exec_time, 2)
                })

            # Validasi jika NIM kosong
            if not nim:
                pbar.write(f"[SKIP] URL tanpa NIM → {url}")
                skip_count += 1
                save_log("SKIPPED", "NIM Kosong", exec_time=time.time() - repo_start_time)
                pbar.update(1)
                continue

            # Bersihkan URL jika ada /tree/main atau branch lain
            clean_url = url.split("/tree/")[0] if "/tree/" in url else url

            # Cek jika folder tujuan sudah ada
            if os.path.exists(target_path):
                pbar.write(f"[SKIP] {nim} sudah ada di direktori.")
                skip_count += 1
                py, ipynb = count_code_files(target_path)
                module = count_modules(target_path)

                total_py_files += py
                total_ipynb_files += ipynb
                save_log(status="SKIPPED", message="Folder sudah ada", module=module, py=py, ipynb=ipynb, repo_size=get_path_size(target_path, unit="kb"), exec_time=time.time() - repo_start_time)
                pbar.update(1)
                continue

            try:
                auth_url = add_token(clean_url, TOKEN_GIT)
                # Eksekusi Git Clone dengan kedalaman dangkal (--depth 1) demi efisiensi RAM
                subprocess.run(
                    [
                        "git", "-c", "core.protectNTFS=false",
                        "clone", "--depth", "1", auth_url, target_path
                    ],
                    check=True, timeout=900,
                    stdout=subprocess.DEVNULL, stderr=subprocess.PIPE
                )

                # Hapus berkas non-kode (hanya menyisakan aset .py dan .ipynb)
                keep_only_code_files(target_path)
                
                # Hitung file
                py, ipynb = count_code_files(target_path)
                module = count_modules(target_path)
                total_py_files += py
                total_ipynb_files += ipynb
                success_count += 1
                
                save_log(status="SUCCESS", message="Clone berhasil", module=module, py=py, ipynb=ipynb, repo_size=get_path_size(target_path, unit="kb"), exec_time=time.time() - repo_start_time)

            except subprocess.CalledProcessError as e:
                error_msg = e.stderr.decode(errors="ignore").strip()
                # Deteksi khusus invalid path
                if "invalid path" in error_msg.lower():
                    custom_msg = "Invalid path Windows / nama file tidak kompatibel"
                    pbar.write(
                        f"⚠️ Repo {nim} gagal checkout karena path tidak valid di Windows")
                    fail_count += 1
                    save_log("FAILED", custom_msg, exec_time=time.time() - repo_start_time)
                else:
                    pbar.write(
                        f"❌ Gagal clone {url}\n"
                        f"    -> {error_msg}"
                    )
                    fail_count += 1
                    save_log("FAILED", error_msg, exec_time=time.time() - repo_start_time)

                # Bersihkan folder jika gagal
                if os.path.exists(target_path):
                    shutil.rmtree(target_path, onerror=remove_readonly)

            except subprocess.TimeoutExpired:
                pbar.write(
                    f"❌ Gagal clone {url}\n"
                    f"    -> Timeout Clone"
                )
                if os.path.exists(target_path):
                    shutil.rmtree(target_path, onerror=remove_readonly)

                fail_count += 1
                save_log("FAILED", "Timeout Clone", exec_time=time.time() - repo_start_time)

            except Exception as e:
                pbar.write(
                    f"❌ Error tidak terduga pada {url}\n"
                    f"    -> {str(e)}"
                )
                if os.path.exists(target_path):
                    shutil.rmtree(target_path, onerror=remove_readonly)

                fail_count += 1
                save_log("FAILED", str(e), exec_time=time.time() - repo_start_time)

            finally:
                pbar.update(1)
            
    return (total_start_time)


# --- EKSEKUSI / RUN PROSES CLONE ---
print("=" * 60)
print(f"{'PROSES CLONING REPOSITORY':^60}")
print("-" * 60)
dataset = load_dataset(INPUT_GITHUB)
print(f"Total input URL: {len(dataset)}\n")
start_time = run_cloning_process(dataset)

execution_time = save_execution_time(start_time=start_time, stage="1_Cloning_Repository", total_mahasiswa=count_students(DIRS['RAW']), total_file=count_all_files(DIRS['RAW'])['total_files'], total_size_kb=get_path_size(DIRS['RAW']))

# Membuat DataFrame dari list hasil
df_results = pd.DataFrame(results)
df_results.to_excel(RESULTS['CLONE_REPORT'], index=False)

display(HTML("<h3> RINGKASAN PROSES CLONING </h3>"))
# Verifikasi jumlah folder fisik yang ada di direktori RAW
repo_in_dir = len([d for d in os.listdir(DIRS['RAW']) if os.path.isdir(os.path.join(DIRS['RAW'], d))])

print("-"*60)
print(f"Total Mahasiswa Terkumpul       : {repo_in_dir}")
print(f"Status Berhasil                 : {success_count}")
print(f"Status Gagal                    : {fail_count}")
print(f"Status Dilewati (Skip)          : {skip_count}")
print(f"Total Waktu Eksekusi            : {execution_time} s")
print(f"Total Size Dataset              : {get_path_size(DIRS['RAW'], unit='mb')} MB")
print('-'*50)
print(f"Total File Python (.py)         : {total_py_files}")
print(f"Total File Notebook (.ipynb)    : {total_ipynb_files}")
print(
    f"Total Keseluruhan File          : {count_all_files(DIRS['RAW'])['total_files']}")

print(f"Laporan Clone disimpan di       : {RESULTS['CLONE_REPORT']}")
display(df_results.head(10))
print("-"*60)
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Diproses pada {timestamp}")

                 PROSES CLONING REPOSITORY                  
------------------------------------------------------------
Total input URL: 57



Cloning Repos:   0%|          | 0/57 [00:00<?, ?repo/s]

❌ Gagal clone https://github.com/fajrulsantoso/244107023010_ML_2025
    -> Cloning into 'D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\00_Raw\244107023010'...
error: unable to create file JS08 /JS08: No such file or directory
fatal: unable to checkout working tree
You can inspect what was checked out with 'git status'
and retry with 'git restore --source=HEAD :/'
❌ Gagal clone https://github.com/Oktavian19/2341720117_ML_2025
    -> Cloning into 'D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\00_Raw\2341720117'...
remote: Repository not found.
fatal: repository 'https://github.com/Oktavian19/2341720117_ML_2025/' not found
❌ Gagal clone https://github.com/KevinASaputra/2341720017_MachineLearning_2025
    -> Cloning into 'D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\00_Raw\2341720017'...
remote: Repository not found.
fatal: repository 'https://github.com/KevinASaputra/2341720017_MachineLearning_2025/' not found
❌ Gagal clone https://github.com/alfbrynn/2341720025_ML_2025

------------------------------------------------------------
Total Mahasiswa Terkumpul       : 53
Status Berhasil                 : 53
Status Gagal                    : 4
Status Dilewati (Skip)          : 0
Total Waktu Eksekusi            : 2116.14 s
Total Size Dataset              : 3552.46 MB
--------------------------------------------------
Total File Python (.py)         : 72
Total File Notebook (.ipynb)    : 2732
Total Keseluruhan File          : 2804
Laporan Clone disimpan di       : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\output(2)\01_clone_report.xlsx


,nim,url,status,message,module,py_files,ipynb_files,total_files,repo_size_kb,execution_time_sec
0,2341720040,https://github.com/AlexanderDev2004/2341720040...,SUCCESS,Clone berhasil,15,5,50,55,12069.31,19.25
1,2341720131,https://github.com/annisaeka123/2341720131_ML_...,SUCCESS,Clone berhasil,13,2,48,50,14926.99,15.12
2,2341720070,https://github.com/annisakrnn/2341720070_ML_2025,SUCCESS,Clone berhasil,14,1,53,54,14963.62,21.89
3,2341720153,https://github.com/AqsaHerryPrastyo/2341720153...,SUCCESS,Clone berhasil,12,0,52,52,13229.81,21.95
4,2241720092,https://github.com/Katakon17/2241720092_ML_2025,SUCCESS,Clone berhasil,5,0,22,22,4689.61,5.09
5,2341720187,https://github.com/4rdnac/2341720187_ML_2025,SUCCESS,Clone berhasil,15,1,51,52,15619.11,13.08
6,2341720144,https://github.com/DanendraPassadhi/2341720144...,SUCCESS,Clone berhasil,15,1,54,55,13554.52,93.22
7,2341720041,https://github.com/dedybayu/2341720041_ML_2025,SUCCESS,Clone berhasil,15,2,52,54,13309.62,40.04
8,2341720111,https://github.com/ekyaaa/2341720111_ML_2025,SUCCESS,Clone berhasil,13,0,51,51,14963.43,9.29
9,2341720218,https://github.com/faishal-ai/machine-learning...,SUCCESS,Clone berhasil,0,0,11,11,7598.59,3.67


------------------------------------------------------------
Diproses pada 2026-06-10 02:38:52


## 2. Normalisasi Struktur Direktori


In [22]:
# HELPER FUNCTIONS (NORMALIZATION)
# ------------------------------------------------------------------------------
log_records = []
def write_log(nim, action, detail, status="SUCCESS", message=""):
    log_records.append({
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "nim": nim,
        "action": action,
        "detail": detail,
        "status": status,
        "message": message
    })

def identify_module(text):
    text = text.upper()

    if "PBL" in text:
        return "pbl"
    if re.search(r"\bUTS\b|UJIAN\s*TENGAH\s*SEMESTER", text):
        return "uts"
    if re.search(r"\bUAS\b|UJIAN\s*AKHIR\s*SEMESTER", text):
        return "uas"
    if re.search(r"KUIS|QUIZ", text):
        return "kuis"
    if re.search(r"KELOMPOK|GROUP", text):
        return "kelompok"

    # JS06p, JS06P, dst → deteksi huruf suffix setelah angka
    # Contoh: JS06p/P → js07, JS06a → js07, JS06b → js08, dst.
    match = re.search(
        r"(JS|JOBSHEET|PERTEMUAN|SESI|MODUL|MODULE|PRAKTIKUM)\s*0?(\d+)([A-Z]?)", text)
    if match:
        base_num = int(match.group(2))
        suffix   = match.group(3)
        if suffix:
            if suffix == 'P':  # Jika Pengayaan (P), naik 1 tingkat
                offset = 1
            else:
                offset = ord(suffix) - ord('A') + 1
            return f"js{base_num + offset:02d}"
        return f"js{base_num:02d}"

    return None

def _p_num(n):
    """Format nomor praktikum: p01–p09, p10, p11, dst."""
    return f"p{n:02d}" if n < 10 else f"p{n}"

def normalize_filename(file):
    name = file.upper()
    ext  = os.path.splitext(file)[1].lower()

    if "PBL" in name:
        return f"pbl{ext}"
    if re.search(r"KELOMPOK|GROUP", name):
        return f"kelompok{ext}"
    if re.search(r"\bUTS\b|UJIAN\s*TENGAH\s*SEMESTER", name):
        return f"uts{ext}"
    if re.search(r"\bUAS\b|UJIAN\s*AKHIR\s*SEMESTER", name):
        return f"uas{ext}"
    if re.search(r"KUIS|QUIZ", name):
        return f"kuis{ext}"
    if re.search(r"(TP|TG|TUGAS)", name):
        return f"tp{ext}"

    # File praktikum P1, P2, dll tidak diekstrak lagi dari nama aslinya, 
    # melainkan diserahkan penuh ke "fallback counter" agar DIJAMIN selalu mulai dari p01.
    return None 

def normalize_filename_with_context(file, module, existing_files_in_target):
    # Aturan Khusus: Jika nama file mengandung 'MODUL', bypass semua logika rename!
    if "MODUL" in file.upper():
        return None  # Mengembalikan None di sini berarti nama file tidak akan diubah

    result = normalize_filename(file)
    if result is not None:
        return result

    # Fallback: Jika bukan PBL/Modul/Tugas dan modul dikenali (jsXX),
    # Berikan nama berurutan (p01, p02, dst) sesuai ketersediaan.
    if module and re.match(r"js\d+", module):
        ext = os.path.splitext(file)[1].lower()
        counter = 1
        while True:
            candidate = f"{_p_num(counter)}{ext}"
            if candidate not in existing_files_in_target:
                return candidate
            counter += 1

    return None  # Nama file tidak dikenali dan tidak perlu diubah

# ANONYMIZE STUDENT FOLDERS
def anonymize_student_folders(source_dir, target_dir=None, mapping_excel="mapping_anonim.xlsx"):
    """
    Anonimisasi folder mahasiswa.
    Mode:
    1. source_dir != target_dir -> copy dataset ke folder baru
    2. source_dir == target_dir -> rename langsung pada folder asal (in-place)
    """
    if target_dir is None:
        target_dir = source_dir

    same_directory = os.path.abspath(source_dir) == os.path.abspath(target_dir)
    student_folders = sorted([f for f in os.listdir(source_dir) if os.path.isdir(os.path.join(source_dir, f))])
    mapping_data = []

    # MODE 1 : RENAME LANGSUNG (IN-PLACE)
    if same_directory:
        temp_mapping = {}

        for idx, student_id in enumerate(student_folders, start=1):
            anonymous_id = f"MHS{idx:03d}"
            temp_name = f"__TEMP_{idx:04d}__"
            os.rename(os.path.join(source_dir, student_id), os.path.join(source_dir, temp_name))
            temp_mapping[temp_name] = {"original": student_id, "anonymous": anonymous_id}

        for temp_name, info in temp_mapping.items():
            os.rename(os.path.join(source_dir, temp_name), os.path.join(source_dir, info["anonymous"]))
            mapping_data.append({"NIM_Asli": info["original"], "Kode": info["anonymous"]})

    # MODE 2 : COPY KE DIREKTORI BARU
    else:
        os.makedirs(target_dir, exist_ok=True)
        for idx, student_id in enumerate(student_folders, start=1):
            anonymous_id = f"MHS{idx:03d}"
            shutil.copytree(os.path.join(source_dir, student_id), os.path.join(target_dir, anonymous_id))
            mapping_data.append({"NIM_Asli": student_id, "Kode": anonymous_id})

    # SIMPAN MAPPING
    mapping_df = pd.DataFrame(mapping_data)
    sheet = "mapping_anonim"
    if os.path.exists(mapping_excel):
        with pd.ExcelWriter(mapping_excel, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
            mapping_df.to_excel(writer, sheet_name=sheet, index=False)
            
            # Mengatur format kolom NIM agar menjadi Teks (mencegah notasi ilmiah)
            worksheet = writer.sheets[sheet]
            for cell in worksheet["A"][1:]: # Skip header
                cell.number_format = "@"
    else:
        with pd.ExcelWriter(mapping_excel, engine="openpyxl") as writer:
            mapping_df.to_excel(
                writer,
                sheet_name=sheet,
                index=False
            )
            worksheet = writer.sheets[sheet]
            for cell in worksheet["A"][1:]:
                cell.number_format = "@"

    print(f"{'Mapping Excel':<20} : {mapping_excel} (sheet name = {sheet})")
    if not same_directory:
        print(f"{'Dataset Anonim':<20} : {target_dir}")
    return mapping_df

def remove_readonly(func, path, _):
    os.chmod(path, stat.S_IWRITE)
    func(path)

def detect_module_from_path(root, file):
    # DIBALIK (reversed): Cek folder paling dalam terlebih dahulu!
    # Agar path Repo/JS06/js06p menangkap js06p (menjadi js07) dan tidak berhenti di JS06.
    for part in reversed(root.split(os.sep)):
        module = identify_module(part)
        if module:
            return module
    return identify_module(file)

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Diproses pada {timestamp}")

Diproses pada 2026-06-10 02:38:55


In [23]:

# CORE LOGIC: NORMALISASI REPOSITORY
# ------------------------------------------------------------------------------
def normalize_repository(repo_path, OUTPUT_DIR):
    nim = os.path.basename(repo_path)
    target_repo_root = os.path.join(OUTPUT_DIR, nim)
    removed_file = 0

    if os.path.exists(target_repo_root):
        shutil.rmtree(target_repo_root, onerror=remove_readonly)
    os.makedirs(target_repo_root, exist_ok=True)

    unclassified_dir = os.path.join(target_repo_root, "unclassified")
    os.makedirs(unclassified_dir, exist_ok=True)

    remove_list = {".idea", ".vscode",".env", ".venv", "env", "venv", "__pycache__", ".git", "colab lama", ".ipynb_checkpoints", "dist", "node_modules", "build"}
    files_to_move = []

    # SCAN SOURCE REPOSITORY
    for root, dirs, files in os.walk(repo_path):
        for d in dirs[:]:
            if d in remove_list:
                removed_path = os.path.join(root, d)
                # Defensive check untuk count_all_files
                removed_count = count_all_files(
                    removed_path)["total_files"] if 'count_all_files' in globals() else 0
                removed_file += removed_count
                write_log(nim, "REMOVE_FOLDER", removed_path, "SUCCESS", f"removed_files={removed_count}")

        dirs[:] = [d for d in dirs if d not in remove_list]
        files.sort()  # Urutkan alfabetis agar sekuensial konsisten

        for file in files:
            full_path = os.path.join(root, file)
            if file.endswith((".py", ".ipynb")):
                files_to_move.append((root, file, full_path))
            else:
                write_log(nim, "SKIP_NON_CODE", full_path)
                removed_file += 1

    # MOVE TO NORMALIZED
    for root, original_name, full_path in files_to_move:
        module = detect_module_from_path(root, original_name)
        module_name = module if module else "unclassified"
        target_subfolder = os.path.join(target_repo_root, module) if module else unclassified_dir
        os.makedirs(target_subfolder, exist_ok=True)
        
        existing_files = set()
        prefix = f"{nim}_{module_name}_"
        for f in os.listdir(target_subfolder):
            if f.startswith(prefix):
                existing_files.add(f[len(prefix):])
            else:
                existing_files.add(f)
                
        new_name = normalize_filename_with_context(original_name, module, existing_files) or original_name        
        norm_status = "RENAMED" if new_name != original_name else "UNCHANGED"

        if new_name != original_name:
            write_log(nim, "RENAME_FILE", f"{original_name} → {new_name}")

        new_name = f"{nim}_{module_name}_{new_name}"
        target_file_path = os.path.join(target_subfolder, new_name)

        # HANDLE DUPLICATE
        if os.path.exists(target_file_path):
            base, ext = os.path.splitext(new_name)
            counter = 2
            while os.path.exists(os.path.join(target_subfolder, f"{base}{counter}{ext}")):
                counter += 1
            new_name = f"{base}{counter}{ext}"
            target_file_path = os.path.join(target_subfolder, new_name)
            write_log(nim, "DUPLICATE_HANDLE", f"{original_name} → {new_name}")
                    
        try:
            shutil.copy2(full_path, target_file_path)
            write_log(nim, "MOVE_FILE",
                      f"{original_name} → {module or 'unclassified'} | "
                      f"hasil: {new_name} [{norm_status}]",
                      "SUCCESS")
            # Hitung nilai fisik data
            file_loc = count_loc(target_file_path)
        except Exception as e:
            write_log(nim, "MOVE_FILE", original_name, "FAILED", str(e))

    return removed_file


# ------------------------------------------------------------------------------
print("=" * 60)
print(f"{' PROSES NORMALISASI DIREKTORI ':^60}")
print("-" * 60)
normalize_start = start_timer()
INPUT_DIR = DIRS['ANON']
OUTPUT_DIR = DIRS['NORM']
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR, onerror=remove_readonly)
os.makedirs(OUTPUT_DIR, exist_ok=True)
mapping_df = anonymize_student_folders(DIRS['RAW'], INPUT_DIR, mapping_excel=RESULTS['NORM_REPORT'])
raw_repos = [
    r for r in os.listdir(INPUT_DIR)
    if os.path.isdir(os.path.join(INPUT_DIR, r))
]

total_processed = 0
total_removed_files = 0
log_records = []

for repo in tqdm(raw_repos, desc="Normalizing Repositories", unit="repo"):
    repo_path = os.path.join(INPUT_DIR, repo)
    try:
        removed_file = normalize_repository(repo_path, OUTPUT_DIR)
        total_removed_files += removed_file
        total_processed += 1
    except Exception as e:
        tqdm.write(f"[ERROR] Gagal proses {repo}: {str(e)}")
        write_log(repo, "NORMALIZE_REPO", "Global Error", "FAILED", str(e))

df_log = pd.DataFrame(log_records)
sheet = "normalisasi"
if 'NORM_REPORT' in RESULTS:
    if os.path.exists(RESULTS['NORM_REPORT']):
        with pd.ExcelWriter(RESULTS['NORM_REPORT'], engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
            df_log.to_excel(
                writer,
                sheet_name=sheet,
                index=False
            )
    else:
        with pd.ExcelWriter(RESULTS['NORM_REPORT'], engine="openpyxl", mode="w") as writer:
            df_log.to_excel(
                writer,
                sheet_name=sheet,
                index=False
            )

print(f"Berhasil memproses {total_processed} repository.")

total_mhs = count_students(df_log)
total_output = count_all_files(OUTPUT_DIR)[
    'total_files'] if 'count_all_files' in globals() else "N/A"
execution_time = save_execution_time(normalize_start, "2_Normalisasi", total_mahasiswa=total_mhs, total_file=total_output, total_size_kb=get_path_size(OUTPUT_DIR))
print("=" * 60)

# SUMMARY & REPORTING
# ── Hitung statistik ─────────────────────────────────────────
df_moved = df_log[df_log["action"] == "MOVE_FILE"].copy()
df_moved["norm_status"] = df_moved["detail"].str.extract(r"\[(\w+)\]$")

total_input = count_all_files(
    DIRS['RAW'])['total_files'] if 'count_all_files' in globals() else "N/A"

total_renamed = (df_moved["norm_status"] == "RENAMED").sum()
total_unchanged = (df_moved["norm_status"] == "UNCHANGED").sum()
total_failed = (df_log["status"] == "FAILED").sum()
total_success = (df_moved["status"] == "SUCCESS").sum()

display(HTML("<h3> 📊 RINGKASAN HASIL NORMALISASI DIREKTORI & FILE </h3>"))
W = 75
print("=" * W)
print(f"  {'Waktu Eksekusi':<35}: {execution_time:>6} detik")
print(f"  {'Total File Input':<35}: {total_input:>6}")
print(f"  {'Total File Output':<35}: {total_output:>6}")
print(f"  {'Total File Excluded / Dihapus':<35}: {total_removed_files:>6}")
print(f"  {'Lokasi Output':<35}: {OUTPUT_DIR}")
print(f"  {'Lokasi File Excel':<35}: {RESULTS['NORM_REPORT']}")
print("-" * W)
print(f"  {'File Dipindah — SUCCESS':<35}: {total_success:>6}")
print(f"    {'↳ Berhasil Di-rename':<33}: {total_renamed:>6}")
print(f"    {'↳ Nama Tidak Berubah':<33}: {total_unchanged:>6}")
print(f"  {'File/Aksi FAILED':<35}: {total_failed:>6}")
print("-" * W)

# Breakdown per aksi
print(f"{'  BREAKDOWN AKSI:':{W}}")
action_counts = df_log["action"].value_counts()
for action, count in action_counts.items():
    print(f"        {action:<35}: {count:>6}")

print("=" * W)

# ── Tabel 1: Sampel File yang Dipindah ───────────────────────
print(f"\n{'─'*W}")
print(f"  📁 DETAIL FILE YANG DINORMALISASI ({len(df_moved)} file)")
print(f"{'─'*W}")

df_display = df_moved[["nim", "detail", "status"]].copy()
df_display.columns = ["NIM / Repo", "Keterangan Pindah & Rename", "Status"]
df_display = df_display.reset_index(drop=True)
df_display.index += 1

if len(df_display) > 5:
    display(df_display.head(5))
    print(
        f"  ... dan {len(df_display) - 5} baris lainnya. (Lihat file Excel untuk detail lengkap)")
else:
    display(df_display)

# ── Tabel 2: File FAILED saja ────────────────────────────────
df_failed = df_log[df_log["status"] == "FAILED"]
if not df_failed.empty:
    print(f"\n{'─'*W}")
    print(f"  ⚠ FILE / AKSI GAGAL ({len(df_failed)} item)")
    print(f"{'─'*W}")
    df_fail_display = df_failed[["nim", "action", "detail", "message"]].copy()
    df_fail_display.columns = ["NIM / Repo", "Aksi", "Detail", "Pesan Error"]
    df_fail_display = df_fail_display.reset_index(drop=True)
    df_fail_display.index += 1
    display(df_fail_display)
else:
    print(f"\n  ✅ Tidak ada file yang gagal dinormalisasi.")

if mapping_df is not None:
    print("\n[PREVIEW] Mapping NIM Asli ke Kode Anonim")
    print("-" * 70)
    display(mapping_df.head())
else:
    print("⚠️ Tidak ada folder mahasiswa yang ditemukan untuk dianonimkan.")

print(f"\n{'-'*W}")
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"  Selesai Diproses pada: {timestamp}")
print("=" * W)

                PROSES NORMALISASI DIREKTORI                
------------------------------------------------------------
Mapping Excel        : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\output(2)\02_normalization_report.xlsx (sheet name = mapping_anonim)
Dataset Anonim       : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\01_Raw_Anon


Normalizing Repositories:   0%|          | 0/53 [00:00<?, ?repo/s]

Berhasil memproses 53 repository.

[SELESAI] 2_Normalisasi | 464.69 s | 2804 file


  Waktu Eksekusi                     : 464.69 detik
  Total File Input                   :   2804
  Total File Output                  :   2804
  Total File Excluded / Dihapus      :      0
  Lokasi Output                      : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\02_Normalized
  Lokasi File Excel                  : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\output(2)\02_normalization_report.xlsx
---------------------------------------------------------------------------
  File Dipindah — SUCCESS            :   2804
    ↳ Berhasil Di-rename             :   2511
    ↳ Nama Tidak Berubah             :    293
  File/Aksi FAILED                   :      0
---------------------------------------------------------------------------
  BREAKDOWN AKSI:                                                          
        MOVE_FILE                          :   2804
        RENAME_FILE                        :   2511
        DUPLICATE_HANDLE                   :     85

─────────────────

,NIM / Repo,Keterangan Pindah & Rename,Status
1,MHS001,Uts.ipynb → uts | hasil: MHS001_uts_uts.ipynb ...,SUCCESS
2,MHS001,P1_JS13.ipynb → js13 | hasil: MHS001_js13_p01....,SUCCESS
3,MHS001,P2_JS13.ipynb → js13 | hasil: MHS001_js13_p02....,SUCCESS
4,MHS001,P3_JS13.ipynb → js13 | hasil: MHS001_js13_p03....,SUCCESS
5,MHS001,TP_JS13.ipynb → js13 | hasil: MHS001_js13_tp.i...,SUCCESS


  ... dan 2799 baris lainnya. (Lihat file Excel untuk detail lengkap)

  ✅ Tidak ada file yang gagal dinormalisasi.

[PREVIEW] Mapping NIM Asli ke Kode Anonim
----------------------------------------------------------------------


,NIM_Asli,Kode
0,2241720092,MHS001
1,2341720003,MHS002
2,2341720005,MHS003
3,2341720007,MHS004
4,2341720009,MHS005



---------------------------------------------------------------------------
  Selesai Diproses pada: 2026-06-10 02:46:40


## 3. Convert file ke .py


In [8]:
# HELPER FUNCTIONS (CONVERTING)
# ------------------------------------------------------------------------------

# LOG SYSTEM
# Menyimpan catatan hasil konversi setiap file
convert_logs = []
def write_convert_log(nim, module, source, target, file_type, status, path_target_file, message=""):
    """
    Mencatat histori konversi dan validasi berkas ke dalam log.
    """
    convert_logs.append({
        "NIM": nim,
        "modul": module,
        "source_file": source,
        "target_file": target,
        "file_type": file_type,
        "status": status,
        "path_target_file": path_target_file,
        "message": message
    })

# VALIDASI SYNTAX
# Memeriksa apakah string kode Python memiliki kesalahan sintaksis secara pasif
def check_syntax_error(code_string):
    """
    Menguji validitas kode menggunakan parser AST bawaan Python.
    """
    try:
        ast.parse(code_string)
        return True, None
    except SyntaxError as e:
        return False, f"SyntaxError: {str(e)}"
    except Exception as e:
        return False, f"OtherError: {str(e)}"
    
# PEMBERSIHAN MAGIC COMMAND
# Menghapus sintaks khusus Jupyter (%, !, %%) yang tidak valid untuk kode Python
def clean_magic_commands(code):
    """
    Pola yang dihapus:
        %...   → line magic       (misal %matplotlib inline)
        !...   → shell command    (misal !pip install ...)
        %%...  → cell magic       (misal %%time)
    """
    code = re.sub(r"^\s*%.*$",  "", code, flags=re.MULTILINE)  # Line magic
    code = re.sub(r"^\s*!.*$",  "", code, flags=re.MULTILINE)  # Shell command
    code = re.sub(r"^\s*%%.*$", "", code, flags=re.MULTILINE)  # Cell magic
    return code


# MENGAMBIL NIM DAN NAMA MODUL
# Menguraikan path relatif untuk mendapatkan NIM dan nama modul
# berdasarkan struktur folder: <base_folder>/<nim>/<module>/
def extract_nim_module(root_path, base_folder):
    parts = os.path.relpath(root_path, base_folder).split(os.sep)

    nim = parts[0] if len(parts) >= 1 else ""
    module = parts[1] if len(parts) >= 2 else ""

    return nim, module


# KONVERSI NOTEBOOK KE PYTHON
def convert_ipynb_to_py(ipynb_path, py_path):
    """
    Mengkonversi file Jupyter Notebook (.ipynb) menjadi file Python (.py).
    Hanya code cell yang diekstrak; markdown dan output diabaikan.
    """
    try:
        with open(ipynb_path, "r", encoding="utf-8") as f:
            notebook = json.load(f)

        code_cells = []
        for cell in notebook.get("cells", []):

            if cell.get("cell_type") != "code":
                continue  # Lewati markdown cell dan raw cell

            source = "".join(cell.get("source", []))
            source = clean_magic_commands(source)
            code_cells.append(source)

        if not code_cells:
            return "FAILED", "Notebook tidak memiliki code cell"
        
        # Gabungkan semua code cell dengan pemisah baris kosong
        full_code = "\n\n".join(code_cells)
        
        # cek validitas syntax sebelum menyimpan
        is_syntax_valid, syntax_error_msg = check_syntax_error(full_code)
        if not is_syntax_valid:
            return "SYNTAX_ERR", syntax_error_msg

        # Gabungkan semua code cell dengan pemisah baris kosong
        with open(py_path, "w", encoding="utf-8") as f:
            f.write(full_code)

        return "SUCCESS", None

    except Exception as e:
        return "FAILED", str(e)


timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Diproses pada {timestamp}")

Diproses pada 2026-06-25 08:58:31


In [15]:
# EXECUTION: CONVERT DATASET
# ------------------------------------------------------------------------------

INPUT_DATASET = DIRS['NORM']
OUTPUT_DATASET = DIRS['CONV']
OUTPUT_LOG    = RESULTS['CONV_REPORT']
if os.path.exists(OUTPUT_DATASET):
    shutil.rmtree(OUTPUT_DATASET)
os.makedirs(OUTPUT_DATASET, exist_ok=True)

converted_success = 0
converted_syntax_err = 0
converted_failed = 0
copy_success = 0
copy_syntax_err = 0
copy_failed = 0
convert_logs = []  # Reset log sebelum mulai


# TRACKING DUPLICATE
seen_targets = {}          # target_path → source_path pertama
duplicate_overwrite = []   # list konflik

# Kumpulkan semua file dari folder hasil normalisasi
all_files = []
for root, dirs, files in os.walk(INPUT_DATASET):
    for file in files:
        all_files.append((root, file))

print("="*60)
print(f"{' MEMULAI KONVERSI KE .PY ':^60}")
print("-"*60)
start = start_timer()
total_loc = 0
pbar = tqdm(all_files, total=len(all_files), desc="Converting", unit="file")

# Looping untuk convert
for root, file in pbar:
    nim, module = extract_nim_module(root, INPUT_DATASET)

    # Siapkan folder tujuan di DIRS['CONV']
    relative_path = os.path.relpath(root, INPUT_DATASET)
    new_root = os.path.join(OUTPUT_DATASET, relative_path)
    os.makedirs(new_root, exist_ok=True)

    source_path = os.path.join(root, file)

    # 1. Jika file sudah .py, cukup copy saja
    if file.endswith(".py"):
        target_path = os.path.join(new_root, file)
        
        # Deteksi Konflik Duplikasi Berkas
        if target_path in seen_targets:
            duplicate_overwrite.append((seen_targets[target_path], source_path, target_path))
            write_convert_log(nim, module, file, new_name, "ipynb", "OVERWRITE", target_path, "Terjadi duplikasi berkas")
            continue
        else:
            seen_targets[target_path] = source_path
            
        try:
            # Baca isi file .py untuk dicek syntax-nya
            with open(source_path, "r", encoding="utf-8") as f:
                py_code = f.read()
            
            is_syntax_valid, syntax_error_msg = check_syntax_error(py_code)
            if not is_syntax_valid:
                copy_syntax_err += 1
                write_convert_log(nim, module, file, file, "py", "SYNTAX_ERR", target_path, syntax_error_msg)
            else:
                shutil.copy2(source_path, target_path)
                copy_success += 1
                
                write_convert_log(nim, module, file, file, "py", "SUCCESS", target_path)
                
        except Exception as e:
            copy_failed += 1
            write_convert_log(nim, module, file, file, "py", "FAILED", target_path, str(e))

    # 2. Jika file .ipynb, lakukan konversi
    elif file.endswith(".ipynb"):
        new_name = file.replace(".ipynb", ".py")
        target_path = os.path.join(new_root, new_name)

        # Deteksi Konflik Duplikasi Berkas
        if target_path in seen_targets:
            duplicate_overwrite.append((seen_targets[target_path], source_path, target_path))
            write_convert_log(nim, module, file, new_name, "ipynb", "OVERWRITE", target_path, "Terjadi duplikasi berkas")
            continue
        else:
            seen_targets[target_path] = source_path
            
        # Eksekusi konversi sekaligus validasi internal
        status_res, msg_res = convert_ipynb_to_py(source_path, target_path)
        
        if status_res == "SUCCESS":
            converted_success += 1
            write_convert_log(nim, module, file, new_name, "ipynb", "SUCCESS", target_path)
        elif status_res == "SYNTAX_ERR":
            converted_syntax_err += 1
            write_convert_log(nim, module, file, new_name, "ipynb", "SYNTAX_ERR", target_path, msg_res)
        else:
            converted_failed += 1
            write_convert_log(nim, module, file, new_name, "ipynb", "FAILED", target_path, msg_res)
        
    pbar.set_postfix_str(
        f"success={converted_success} convert_fail={converted_failed} copy_fail={copy_failed}")

pbar.close()

# Simpan Log Lengkap ke Excel
df_convert = pd.DataFrame(convert_logs)
df_convert.to_excel(OUTPUT_LOG, index=False)
total_mhs = df_convert['NIM'].nunique() if not df_convert.empty else 0

# Hitung statistik akhir untuk verifikasi fisik direktori
counts_before = count_all_files(INPUT_DATASET)
counts_after = count_all_files(OUTPUT_DATASET)
total_file = counts_after['total_files']
total_size = get_path_size(OUTPUT_DATASET)

total_time = save_execution_time(start, "3_Convert_to_PY", total_mhs, total_file, total_size)
print(f"\n{' PROSES KONVERSI SELESAI ':=^60}")

#  SUMMARY
total_syntax_err = copy_syntax_err + converted_syntax_err
total_io_fail = copy_failed + converted_failed

display(HTML("<h3> RINGKASAN HASIL KONVERSI DATASET </h3>"))
print("-"*50)
print(f"{'Total Mahasiswa (Repo)':<28} : {total_mhs}")
print(f"{'Total Berkas Masukan':<28} : {counts_before['total_files']} ({counts_before['by_extension']['.ipynb']} ipynb, {counts_before['by_extension']['.py'] if not counts_before['by_extension'].get('.py') is None else 0} py)")
print("-" * 50)
print(f"{'Sukses Diproses':<28} : {copy_success + converted_success}")
print(f"{'Syntax Error':<28} : {total_syntax_err}")
print(f"{'Jumlah overwrite':<28} : {len(duplicate_overwrite)}")
print(f"{'Gagal Diproses':<28} : {total_io_fail}")
print("-" * 50)
print(f"{'Total File Akhir':<28} : {total_file}")
print("="*50)
print(f"Laporan Lengkap Audit Data: {OUTPUT_LOG}")
print("="*50 + "\n")

# Tampilkan cuplikan struktur log data frame
display(df_convert.head(5))
print("-" * 50)
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Diproses pada {timestamp}")

                  MEMULAI KONVERSI KE .PY                   
------------------------------------------------------------


Converting:   0%|          | 0/2804 [00:00<?, ?file/s]


[SELESAI] 3_Convert_to_PY | 144.13 s | 2713 file

================= PROSES KONVERSI SELESAI ==================


--------------------------------------------------
Total Mahasiswa (Repo)       : 53
Total Berkas Masukan         : 2804 (2732 ipynb, 72 py)
--------------------------------------------------
Sukses Diproses              : 2713
Syntax Error                 : 49
Jumlah overwrite             : 15
Gagal Diproses               : 27
--------------------------------------------------
Total File Akhir             : 2713
Laporan Lengkap Audit Data: D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\output(2)\03_conversion_report.xlsx



,NIM,modul,source_file,target_file,file_type,status,path_target_file,message
0,MHS001,js02,MHS001_js02_p01.ipynb,MHS001_js02_p01.py,ipynb,SUCCESS,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,
1,MHS001,js02,MHS001_js02_p02.ipynb,MHS001_js02_p02.py,ipynb,SUCCESS,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,
2,MHS001,js02,MHS001_js02_p03.ipynb,MHS001_js02_p03.py,ipynb,SUCCESS,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,
3,MHS001,js02,MHS001_js02_p04.ipynb,MHS001_js02_p04.py,ipynb,SUCCESS,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,
4,MHS001,js02,MHS001_js02_tp.ipynb,MHS001_js02_tp.py,ipynb,SUCCESS,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,


--------------------------------------------------
Diproses pada 2026-06-14 12:11:04


### contoh validasi syntax

In [12]:
# ==========================
# Demo validasi syntax
# ==========================

file_path = "coba.py"

with open(file_path, "r", encoding="utf-8") as f:
    code = f.read()

is_valid, message = check_syntax_error(code)

if is_valid:
    print("✓ Kode valid, siap diproses menjadi AST.")
else:
    print("✗ Kode tidak valid. Terdapat kesalahan")
    print(message)

✓ Kode valid, siap diproses menjadi AST.


## 4. Data Cleaning


In [39]:
# HELPER FUNCTION (CLEANING)
# --------------------------------------------------------------------------------------------

# HAPUS COMMAND PIP & MAGIC COMMAND LAINNYA
def hapus_pip_commands(kode):
    lines = kode.split('\n')
    lines_bersih = []
    removed = 0
    
    for line in lines:
        stripped = line.strip()
        
        # 1. Menangkap !pip, %pip, ! pip, % pip, pip install, pip3 install
        if re.match(r'^(!|%)?\s*pip3?\b', stripped):
            removed += 1
            continue
            
        # 2. Menangkap format konversi nbconvert (get_ipython)
        if 'get_ipython()' in stripped and ('pip' in stripped) and ('system' in stripped or 'run_line_magic' in stripped):
            removed += 1
            continue
            
        # 3. Menangkap eksekusi shell via os.system untuk pip
        if 'os.system' in stripped and 'pip' in stripped:
            removed += 1
            continue
            
        # Jika bukan command pip, masukkan ke list bersih
        lines_bersih.append(line)
        
    return '\n'.join(lines_bersih), removed


# HAPUS KOMENTAR
def hapus_komentar_fallback(kode):
    lines_bersih = []
    removed = 0

    for line in kode.split('\n'):
        if '#' in line:
            in_string = False
            chars = list(line)

            for i, ch in enumerate(chars):
                if ch in ('"', "'") and (i == 0 or chars[i-1] != '\\'):
                    in_string = not in_string

                if ch == '#' and not in_string:
                    line = line[:i]
                    removed += 1
                    break

        lines_bersih.append(line.rstrip())

    return '\n'.join(lines_bersih), removed


def hapus_komentar(kode):
    try:
        tokens = tokenize.generate_tokens(io.StringIO(kode).readline)
        tokens_bersih = []
        removed = 0

        for t in tokens:
            if t.type == tokenize.COMMENT:
                removed += 1
            else:
                tokens_bersih.append(t)

        return tokenize.untokenize(tokens_bersih), 'TOKENIZE', removed

    except:
        kode_bersih, removed = hapus_komentar_fallback(kode)
        return kode_bersih, 'FALLBACK', removed


# HAPUS DOCSTRING
class DocstringRemover(ast.NodeTransformer):
    def _strip(self, node):
        if (
            node.body
            and isinstance(node.body[0], ast.Expr)
            and isinstance(node.body[0].value, ast.Constant)
            and isinstance(node.body[0].value.value, str)
        ):
            node.body.pop(0)

        if not node.body:
            node.body.append(ast.Pass())

        self.generic_visit(node)
        return node

    visit_Module = visit_FunctionDef = visit_AsyncFunctionDef = visit_ClassDef = _strip


def hapus_docstring_fallback(kode):
    lines = kode.split('\n')
    lines_bersih = []

    in_doc = False
    doc_char = None
    removed = 0

    for line in lines:
        if not in_doc:
            for q in ('"""', "'''"):
                idx = line.find(q)

                if idx != -1:
                    removed += 1
                    closing = line.find(q, idx + 3)

                    if closing != -1:
                        line = line[:idx] + line[closing + 3:]
                    else:
                        line = line[:idx]
                        in_doc = True
                        doc_char = q
                    break

            lines_bersih.append(line.rstrip())

        else:
            if doc_char in line:
                pos = line.find(doc_char)
                line = line[pos + 3:]
                in_doc = False
                doc_char = None

            lines_bersih.append(line.rstrip())

    return '\n'.join(lines_bersih), removed


def hapus_docstring(kode):
    try:
        removed = 0
        tree = ast.parse(kode)
        for node in ast.walk(tree):
            if isinstance(node, (ast.Module, ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                if (
                    node.body
                    and isinstance(node.body[0], ast.Expr)
                    and isinstance(node.body[0].value, ast.Constant)
                    and isinstance(node.body[0].value.value, str)
                ):
                    removed += 1
        tree = DocstringRemover().visit(tree)
        ast.fix_missing_locations(tree)
        
        return ast.unparse(tree), 'AST', removed

    except:
        kode_bersih, removed = hapus_docstring_fallback(kode)
        return kode_bersih, 'FALLBACK', removed


# NORMALISASI WHITESPACE
def normalisasi_whitespace(kode):
    lines = kode.split('\n')
    lines_bersih = []

    for line in lines:
        stripped = line.rstrip()
        if stripped.strip() == '':
            continue

        lines_bersih.append(stripped)

    return '\n'.join(lines_bersih)

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Diproses pada {timestamp}")

Diproses pada 2026-06-10 03:03:37


In [40]:
# EKSEKUSI CLEANING
# ---------------------------------------------------------------------
INPUT_DIR = DIRS['CONV']
OUTPUT_DIR = DIRS['CLEAN']
cleaning_logs = []

total_loc = 0
start = start_timer()

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

all_py_files = []
for root, dirs, files in os.walk(INPUT_DIR):
    for fname in files:
        if fname.endswith('.py'):
            all_py_files.append((os.path.join(root, fname), root))

print(f"{' MULAI CLEANING CODE ':=^70}")

for source_path, root in tqdm(all_py_files, desc="Cleaning Process", unit="file"):
    rel_path = os.path.relpath(root, INPUT_DIR)
    target_dir = os.path.join(OUTPUT_DIR, rel_path)
    os.makedirs(target_dir, exist_ok=True)

    target_path = os.path.join(target_dir, os.path.basename(source_path))

    parts = rel_path.split(os.sep)
    nim = parts[0] if len(parts) >= 1 else "unknown"
    modul = parts[1] if len(parts) >= 2 else "root"
    file_name = os.path.basename(source_path)
    file_id = os.path.splitext(file_name)[0]

    try:
        with open(source_path, 'r', encoding='utf-8', errors='replace') as f:
            kode = f.read()

        loc_before = len(kode.split('\n'))
        blank_before = sum(1 for line in kode.split('\n') if line.strip() == '')

        # 1. Hapus Pip commands terlebih dahulu agar AST tidak error
        kode, pip_removed = hapus_pip_commands(kode)
        
        # 2. Hapus komentar & docstring
        kode, m_komentar, komentar_removed = hapus_komentar(kode)
        kode, m_docstring, docstring_removed = hapus_docstring(kode)
        
        # 3. Terakhir, normalisasi whitespace kosong
        kode = normalisasi_whitespace(kode)

        loc_after = len(kode.split('\n'))
        blank_after = sum(1 for line in kode.split('\n') if line.strip() == '')

        # CEK FILE KOSONG ATAU HANYA BERISI 'pass'
        cek_kode = kode.strip()
        if not cek_kode or cek_kode == 'pass':
            cleaning_logs.append({
                "nim": nim,
                "modul": modul,
                "file": os.path.basename(source_path),
                "method_komentar": m_komentar,
                "method_docstring": m_docstring,
                "status": "REMOVED_EMPTY",
                "loc_before": loc_before,
                "loc_after": 0,
                "pip_removed": pip_removed,
                "comment_removed": komentar_removed,
                "docstring_removed": docstring_removed,
                "blank_lines_removed": blank_before,
                "message": "File dihapus karena kosong atau hanya berisi 'pass'."
            })
            continue  # Loncati proses penulisan file, file ini dibuang

        # Jika kode valid, tulis ke folder CLEAN
        with open(target_path, 'w', encoding='utf-8') as f:
            f.write(kode)

        cleaning_logs.append({
            "nim": nim,
            "modul": modul,
            "file": os.path.basename(source_path),
            "method_komentar": m_komentar,
            "method_docstring": m_docstring,
            "status": "SUCCESS",
            "loc_before": loc_before,
            "loc_after": loc_after,
            "pip_removed": pip_removed,
            "comment_removed": komentar_removed,
            "docstring_removed": docstring_removed,
            "blank_lines_removed": (blank_before - blank_after),
            "message": ""
        })

    except Exception as e:
        cleaning_logs.append({
            "nim": nim,
            "modul": modul,
            "file": os.path.basename(source_path),
            "status": "FAILED",
            "loc_before": 0,
            "loc_after": 0,
            "pip_removed": 0,
            "comment_removed": 0,
            "docstring_removed": 0,
            "blank_lines_removed": 0,
            "message": str(e)
        })
# Simpan Log ke Excel khusus cleaning
df_clean = pd.DataFrame(cleaning_logs)
df_clean.to_excel(RESULTS['CLEAN_REPORT'], index=False)

total_input = count_all_files(INPUT_DIR)['total_files'] if 'count_all_files' in globals() else len(all_py_files)
total_success = len(df_clean[df_clean['status'] == 'SUCCESS'])
total_removed = len(df_clean[df_clean['status'] == 'REMOVED_EMPTY'])
total_failed = len(df_clean[df_clean['status'] == 'FAILED'])
total_pip_removed = df_clean['pip_removed'].sum()

total_mhs = count_students(df_clean)
total_time = save_execution_time(start, "4_Cleaning_Data", total_mhs, total_success, get_path_size(OUTPUT_DIR))
print(f"\n{' PROSES CLEANING SELESAI ':=^70}")

W = 75
display(HTML(f"<h3>🧹 RINGKASAN HASIL CLEANING DATA </h3>"))
print("-"*W)
print(f"{'Input File dari Folder CONV':<40} : {total_input}")
print(f"{'Total File Sukses Dibersihkan':<40} : {total_success}")
print(f"{'Total File Dihapus (Kosong/Hanya Pass)':<40} : {total_removed}")
print(f"{'Total File Gagal Diproses':<40} : {total_failed}")
print(f"{'Total Command Pip/Magic Dihapus':<40} : {total_pip_removed} baris")
print(f"{'Folder Output Bersih':<40} : {OUTPUT_DIR}")
print("-"*W)
print(f"{'File Log Disimpan di':<40} : {RESULTS['CLEAN_REPORT']}")

if total_removed > 0:
    print("\n[!] Beberapa file diabaikan/dihapus karena tidak memiliki baris kode valid setelah cleaning.")

display(df_clean.head(10))

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Diproses pada {timestamp}")

======================== MULAI CLEANING CODE =========================


Cleaning Process:   0%|          | 0/2713 [00:00<?, ?file/s]


[SELESAI] 4_Cleaning_Data | 200.98 s | 2667 file

====================== PROSES CLEANING SELESAI =======================


---------------------------------------------------------------------------
Input File dari Folder CONV              : 2713
Total File Sukses Dibersihkan            : 2667
Total File Dihapus (Kosong/Hanya Pass)   : 46
Total File Gagal Diproses                : 0
Total Command Pip/Magic Dihapus          : 15 baris
Folder Output Bersih                     : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\04_Cleaned
---------------------------------------------------------------------------
File Log Disimpan di                     : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\output(2)\04_cleaning_report.xlsx

[!] Beberapa file diabaikan/dihapus karena tidak memiliki baris kode valid setelah cleaning.


,nim,modul,file,method_komentar,method_docstring,status,loc_before,loc_after,pip_removed,comment_removed,docstring_removed,blank_lines_removed,message
0,MHS001,js02,MHS001_js02_p01.py,TOKENIZE,AST,SUCCESS,69,47,0,5,0,17,
1,MHS001,js02,MHS001_js02_p02.py,TOKENIZE,AST,SUCCESS,26,12,0,5,0,9,
2,MHS001,js02,MHS001_js02_p03.py,TOKENIZE,AST,SUCCESS,40,27,0,2,0,11,
3,MHS001,js02,MHS001_js02_p04.py,TOKENIZE,AST,SUCCESS,32,18,0,5,0,9,
4,MHS001,js02,MHS001_js02_tp.py,TOKENIZE,AST,SUCCESS,120,67,0,30,0,23,
5,MHS001,js03,MHS001_js03_p01.py,TOKENIZE,AST,SUCCESS,75,41,0,9,0,21,
6,MHS001,js03,MHS001_js03_p02.py,TOKENIZE,AST,SUCCESS,31,18,0,4,0,9,
7,MHS001,js03,MHS001_js03_p03.py,TOKENIZE,AST,SUCCESS,36,22,0,10,0,7,
8,MHS001,js03,MHS001_js03_p04.py,TOKENIZE,AST,SUCCESS,17,6,0,4,0,8,
9,MHS001,js03,MHS001_js03_tp.py,TOKENIZE,AST,SUCCESS,219,108,0,44,0,48,


Diproses pada 2026-06-10 03:12:23


## 5. Normalisasi Format Spasi (Black & Autopep8)

### 5a. Black

In [41]:
# ============================================================
# Normalisasi Format Spasi dengan Black
# Tujuan: Memformat semua .py di converted_py menggunakan Black
# Output : dataset\normalized_black\
# File asli di converted_py TIDAK diubah
# ============================================================

import black
from pathlib import Path

# ── Folder output ────────────────────────────────────────────
# BLACK_DIR = os.path.join(DATASET_DIR, 'normalized_black')
# os.makedirs(BLACK_DIR, exist_ok=True)

# ── Konfigurasi Black ────────────────────────────────────────
BLACK_MODE = black.Mode(
    line_length=88,
    string_normalization=False,
    magic_trailing_comma=True,
)

# ── Kumpulkan semua .py dari converted_py ───────────────────
all_py = []
for nim in os.listdir(DIRS['CLEAN']):
    nim_path = os.path.join(DIRS['CLEAN'], nim)
    if not os.path.isdir(nim_path):
        continue
    for root, dirs, files in os.walk(nim_path):
        for fname in files:
            if fname.endswith('.py'):
                src = os.path.join(root, fname)
                rel = os.path.relpath(src, DIRS['CLEAN'])
                all_py.append((src, rel))

print('=' * 70)
print(' Normalisasi dengan Black')
print(f'   Sumber     : {DIRS["CLEAN"]}')
print(f'   Output     : {DIRS["BLACK"]}')
print(f'   Line length: 88 karakter')
print(f'   Total file : {len(all_py)}')
print('=' * 70)

black_ok   = 0
black_fail = 0
black_fail_list = []

total_loc = 0
black_start = start_timer()

for src_path, rel_path in tqdm(all_py, desc='Black'):
    dst_path = os.path.join(DIRS['BLACK'], rel_path)
    os.makedirs(os.path.dirname(dst_path), exist_ok=True)
    
    parts = rel_path.split(os.sep)
    nim = parts[0] if len(parts) >= 1 else "unknown"
    modul = parts[1] if len(parts) >= 2 else "root"
    file_name = os.path.basename(rel_path)
    file_id = os.path.splitext(file_name)[0]

    # Idempotent — skip jika sudah ada
    if os.path.exists(dst_path):
        black_ok += 1
        continue

    # Baca file sumber
    try:
        with open(src_path, 'r', encoding='utf-8', errors='replace') as f:
            original = f.read()
    except Exception as e:
        black_fail += 1
        black_fail_list.append({'file': rel_path, 'error': str(e)})
        continue

    # Format dengan Black
    try:
        formatted = black.format_str(original, mode=BLACK_MODE)
        with open(dst_path, 'w', encoding='utf-8') as f:
            f.write(formatted)
        black_ok += 1
        
    except Exception as e:
        # Syntax error — salin apa adanya agar file tetap ada
        with open(dst_path, 'w', encoding='utf-8') as f:
            f.write(original)
        black_fail += 1        
        black_fail_list.append({'file': rel_path, 'error': str(e)})

# Simpan log gagal
if black_fail_list:
    # log_path = os.path.join(LOG_DIR, 'black_errors.json')
    with open(RESULTS['ERR_BLACK'], 'w', encoding='utf-8') as f:
        json.dump(black_fail_list, f, indent=2, ensure_ascii=False)
    print(f'\n⚠️  Log error disimpan di: {RESULTS["ERR_BLACK"]}')

total_mhs   = count_students(DIRS['BLACK'])
total_file  = count_all_files(DIRS['BLACK'])['total_files']
total_size  = get_path_size(DIRS['BLACK'])

black_total_time = save_execution_time(black_start, "5a_Black_norm", total_mhs, total_file, total_size)

print('=' * 70)
print('📊 Hasil Normalisasi Black')
print('=' * 70)
print(f'  Waktu Eksekusi         : {black_total_time} detik')
print(f'  ✅ Berhasil diformat   : {black_ok}')
print(f'  ❌ Gagal (syntax error): {black_fail}')
print(f'  📂 Output              : {DIRS["BLACK"]}')
print('=' * 70)

 Normalisasi dengan Black
   Sumber     : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\04_Cleaned
   Output     : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\05b_Black
   Line length: 88 karakter
   Total file : 2667


Black:   0%|          | 0/2667 [00:00<?, ?it/s]


[SELESAI] 5a_Black_norm | 586.71 s | 5334 file
📊 Hasil Normalisasi Black
  Waktu Eksekusi         : 586.71 detik
  ✅ Berhasil diformat   : 2667
  ❌ Gagal (syntax error): 0
  📂 Output              : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\05b_Black


### 5b. Autopep8

In [42]:
# ============================================================
# CELL 15 — Normalisasi Format Spasi dengan Autopep8
# Tujuan: Memformat semua .py di converted_py menggunakan Autopep8
# Output : dataset\normalized_autopep8\
# File asli di converted_py TIDAK diubah
# ============================================================

import autopep8

# ── Folder output ────────────────────────────────────────────
# AUTOPEP_DIR = os.path.join(DATASET_DIR, 'normalized_autopep8')
# os.makedirs(AUTOPEP_DIR, exist_ok=True)

print('=' * 70)
print('🔵 Normalisasi dengan Autopep8')
print(f'   Sumber     : {DIRS["CLEAN"]}')
print(f'   Output     : {DIRS["AUTOPEP8"]}')
print(f'   Line length: 79 karakter (PEP8 standar)')
print(f'   Aggressive : level 1')
print(f'   Total file : {len(all_py)}')
print('=' * 70)

autopep_ok   = 0
autopep_fail = 0
autopep_fail_list = []

start = start_timer()
for src_path, rel_path in tqdm(all_py, desc='Autopep8'):

    dst_path = os.path.join(DIRS["AUTOPEP8"], rel_path)
    os.makedirs(os.path.dirname(dst_path), exist_ok=True)

    # Idempotent — skip jika sudah ada
    if os.path.exists(dst_path):
        autopep_ok += 1
        continue

    # Baca file sumber
    try:
        with open(src_path, 'r', encoding='utf-8', errors='replace') as f:
            original = f.read()
    except Exception as e:
        autopep_fail += 1
        autopep_fail_list.append({'file': rel_path, 'error': str(e)})
        continue

    # Format dengan Autopep8
    try:
        formatted = autopep8.fix_code(
            original,
            options={
                'max_line_length': 79,
                'aggressive'     : 1,
            }
        )
        with open(dst_path, 'w', encoding='utf-8') as f:
            f.write(formatted)
        autopep_ok += 1
    except Exception as e:
        # Fallback — salin apa adanya
        with open(dst_path, 'w', encoding='utf-8') as f:
            f.write(original)
        autopep_fail += 1
        autopep_fail_list.append({'file': rel_path, 'error': str(e)})

# Simpan log gagal
if autopep_fail_list:
    # log_path = os.path.join(LOG_DIR, 'autopep8_errors.json')
    with open(RESULTS['ERR_AUTOPEP'], 'w', encoding='utf-8') as f:
        json.dump(autopep_fail_list, f, indent=2, ensure_ascii=False)
    print(f'\n⚠️  Log error disimpan di: {RESULTS["ERR_AUTOPEP"]}')

total_mhs   = count_students(DIRS['AUTOPEP8'])
total_files = count_all_files(DIRS['AUTOPEP8'])['total_files']
total_size  = get_path_size(DIRS['AUTOPEP8'])
pep_time = save_execution_time(start, "5b_Autopep8_Norm", total_mhs, total_files, total_size)
print('=' * 70)
print('📊 Hasil Normalisasi Autopep8')
print('=' * 70)
print(f'  ✅ Berhasil diformat   : {autopep_ok}')
print(f'  ❌ Gagal               : {autopep_fail}')
print(f'  📂 Output              : {DIRS["AUTOPEP8"]}')
print('=' * 70)

🔵 Normalisasi dengan Autopep8
   Sumber     : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\04_Cleaned
   Output     : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\05a_Autopep8
   Line length: 79 karakter (PEP8 standar)
   Aggressive : level 1
   Total file : 2667


Autopep8:   0%|          | 0/2667 [00:00<?, ?it/s]


[SELESAI] 5b_Autopep8_Norm | 1568.96 s | 5334 file
📊 Hasil Normalisasi Autopep8
  ✅ Berhasil diformat   : 2667
  ❌ Gagal               : 0
  📂 Output              : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\05a_Autopep8


### Analisis Diff Black vs Autopep8

In [43]:
# ============================================================
# CELL 16 — Analisis Diff Black vs Autopep8
# Tujuan: Menghitung perubahan per file untuk kedua formatter
#         Hasil disimpan ke df_report untuk diekspor di Cell 14
# ============================================================

import difflib

def count_changed_lines(original: str, formatted: str):
    """
    Hitung jumlah baris yang berubah antara original dan formatted.
    Return: (jumlah_baris_berubah, contoh_sebelum, contoh_sesudah)
    """
    orig_lines = original.splitlines()
    fmt_lines  = formatted.splitlines()
    changed    = 0
    ex_before  = ''
    ex_after   = ''

    matcher = difflib.SequenceMatcher(None, orig_lines, fmt_lines)
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag in ('replace', 'insert', 'delete'):
            changed += max(i2 - i1, j2 - j1)
            # Ambil 1 contoh baris replace yang paling representatif
            if not ex_before and tag == 'replace':
                for k in range(i1, i2):
                    if k < len(orig_lines) and orig_lines[k].strip():
                        ex_before = orig_lines[k][:80]
                        j_idx = j1 + (k - i1)
                        ex_after = fmt_lines[j_idx][:80] if j_idx < len(fmt_lines) else ''
                        break
    return changed, ex_before, ex_after

print('=' * 70)
print('🔍 Menganalisis diff per file...')
print(f'   Total file dianalisis: {len(all_py)}')
print('=' * 70)

records = []

for src_path, rel_path in tqdm(all_py, desc='Analisis diff'):
    nim       = rel_path.split(os.sep)[0]
    fname     = os.path.basename(rel_path)
    subfolder = os.path.dirname(rel_path.split(os.sep, 1)[-1]) if os.sep in rel_path else ''

    path_black = os.path.join(DIRS['BLACK'], rel_path)
    path_auto  = os.path.join(DIRS['AUTOPEP8'], rel_path)

    # ── Baca original ────────────────────────────────────────
    try:
        with open(src_path, 'r', encoding='utf-8', errors='replace') as f:
            orig = f.read()
        orig_lines = len(orig.splitlines())
    except Exception:
        orig = ''
        orig_lines = 0

    # ── Analisis Black ───────────────────────────────────────
    try:
        with open(path_black, 'r', encoding='utf-8', errors='replace') as f:
            blk = f.read()
        blk_lines            = len(blk.splitlines())
        blk_changed, beb, bea = count_changed_lines(orig, blk)
        blk_pct              = round(blk_changed / orig_lines * 100, 1) if orig_lines else 0
        blk_status           = 'Berubah' if blk_changed > 0 else 'Tidak Berubah'
    except Exception:
        blk_lines = blk_changed = blk_pct = 0
        beb = bea = ''
        blk_status = 'Gagal'

    # ── Analisis Autopep8 ────────────────────────────────────
    try:
        with open(path_auto, 'r', encoding='utf-8', errors='replace') as f:
            auto = f.read()
        auto_lines             = len(auto.splitlines())
        auto_changed, aeb, aea = count_changed_lines(orig, auto)
        auto_pct               = round(auto_changed / orig_lines * 100, 1) if orig_lines else 0
        auto_status            = 'Berubah' if auto_changed > 0 else 'Tidak Berubah'
    except Exception:
        auto_lines = auto_changed = auto_pct = 0
        aeb = aea = ''
        auto_status = 'Gagal'

    records.append({
        'NIM'                        : nim,
        'Subfolder'                  : subfolder,
        'Nama File'                  : fname,
        'Baris Original'             : orig_lines,
        'Black — Baris Hasil'        : blk_lines,
        'Black — Baris Berubah'      : blk_changed,
        'Black — % Perubahan'        : blk_pct,
        'Black — Status'             : blk_status,
        'Black — Contoh Sebelum'     : beb,
        'Black — Contoh Sesudah'     : bea,
        'Autopep8 — Baris Hasil'     : auto_lines,
        'Autopep8 — Baris Berubah'   : auto_changed,
        'Autopep8 — % Perubahan'     : auto_pct,
        'Autopep8 — Status'          : auto_status,
        'Autopep8 — Contoh Sebelum'  : aeb,
        'Autopep8 — Contoh Sesudah'  : aea,
    })

df_report = pd.DataFrame(records)

# Preview ringkasan
blk_b  = (df_report['Black — Status']    == 'Berubah').sum()
blk_u  = (df_report['Black — Status']    == 'Tidak Berubah').sum()
blk_f  = (df_report['Black — Status']    == 'Gagal').sum()
auto_b = (df_report['Autopep8 — Status'] == 'Berubah').sum()
auto_u = (df_report['Autopep8 — Status'] == 'Tidak Berubah').sum()
auto_f = (df_report['Autopep8 — Status'] == 'Gagal').sum()

print()
print('=' * 70)
print('📊 Ringkasan Hasil Analisis Diff')
print('=' * 70)
print(f'  {"Metrik":<30} {"Black":>10} {"Autopep8":>10}')
print(f'  {"-"*50}')
print(f'  {"File Berubah":<30} {blk_b:>10} {auto_b:>10}')
print(f'  {"File Tidak Berubah":<30} {blk_u:>10} {auto_u:>10}')
print(f'  {"File Gagal":<30} {blk_f:>10} {auto_f:>10}')
print(f'  {"Rata-rata % Perubahan":<30} '
      f'{df_report["Black — % Perubahan"].mean():>9.1f}% '
      f'{df_report["Autopep8 — % Perubahan"].mean():>9.1f}%')
print('=' * 70)
print(f'  ✅ df_report siap — {len(df_report)} baris. Lanjutkan ke Cell 14.')
print('=' * 70)

🔍 Menganalisis diff per file...
   Total file dianalisis: 2667


Analisis diff:   0%|          | 0/2667 [00:00<?, ?it/s]


📊 Ringkasan Hasil Analisis Diff
  Metrik                              Black   Autopep8
  --------------------------------------------------
  File Berubah                         2656       2365
  File Tidak Berubah                     11        302
  File Gagal                              0          0
  Rata-rata % Perubahan               42.5%      41.3%
  ✅ df_report siap — 2667 baris. Lanjutkan ke Cell 14.


### Export Laporan Perbandingan ke Excel

In [44]:
# ============================================================
# CELL 17 — Export Laporan Perbandingan ke Excel
# Tujuan: Menyimpan df_report ke .xlsx dengan 3 sheet:
#   Sheet 1 — Detail Per File
#   Sheet 2 — Ringkasan per NIM
#   Sheet 3 — Ringkasan Keseluruhan (Black vs Autopep8)
# Output : Skripsi_AST1\normalisasi_perbandingan.xlsx
# ============================================================

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

REPORT_PATH = os.path.join(DIRS['LOGS'], 'normalisasi_perbandingan_black_autopep.xlsx')

# ── Palet warna ──────────────────────────────────────────────
C_HDR_BASE  = '2E7D32'   # hijau tua   — kolom info file
C_HDR_BLACK = '2C2C2C'   # hitam       — kolom Black
C_HDR_AUTO  = '1565C0'   # biru tua    — kolom Autopep8
C_ROW_EVEN  = 'F5F5F5'
C_ROW_ODD   = 'FFFFFF'
C_CHANGED   = 'FFF9C4'   # kuning muda — ada perubahan
C_UNCHANGED = 'E8F5E9'   # hijau muda  — tidak berubah
C_FAIL      = 'FFEBEE'   # merah muda  — gagal

GROUP_COLOR = {'base': C_HDR_BASE, 'black': C_HDR_BLACK, 'auto': C_HDR_AUTO}

def thin_border():
    s = Side(style='thin', color='CCCCCC')
    return Border(left=s, right=s, top=s, bottom=s)

STATUS_FILL = {
    'Berubah'      : PatternFill('solid', fgColor=C_CHANGED),
    'Tidak Berubah': PatternFill('solid', fgColor=C_UNCHANGED),
    'Gagal'        : PatternFill('solid', fgColor=C_FAIL),
}

wb = Workbook()

# ════════════════════════════════════════════════════════════
# SHEET 1 — Detail Per File
# ════════════════════════════════════════════════════════════
ws1 = wb.active
ws1.title = 'Detail Per File'

COLS_DETAIL = [
    ('NIM',                        16, 'base'),
    ('Subfolder',                  30, 'base'),
    ('Nama File',                  28, 'base'),
    ('Baris Original',             15, 'base'),
    ('Black — Baris Hasil',        18, 'black'),
    ('Black — Baris Berubah',      20, 'black'),
    ('Black — % Perubahan',        18, 'black'),
    ('Black — Status',             16, 'black'),
    ('Black — Contoh Sebelum',     45, 'black'),
    ('Black — Contoh Sesudah',     45, 'black'),
    ('Autopep8 — Baris Hasil',     20, 'auto'),
    ('Autopep8 — Baris Berubah',   22, 'auto'),
    ('Autopep8 — % Perubahan',     20, 'auto'),
    ('Autopep8 — Status',          18, 'auto'),
    ('Autopep8 — Contoh Sebelum',  45, 'auto'),
    ('Autopep8 — Contoh Sesudah',  45, 'auto'),
]

for ci, (label, width, grp) in enumerate(COLS_DETAIL, 1):
    c = ws1.cell(row=1, column=ci, value=label)
    c.font      = Font(name='Arial', bold=True, color='FFFFFF', size=10)
    c.fill      = PatternFill('solid', fgColor=GROUP_COLOR[grp])
    c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    c.border    = thin_border()
    ws1.column_dimensions[get_column_letter(ci)].width = width
ws1.row_dimensions[1].height = 32

for ri, rec in enumerate(records, 2):
    bg = PatternFill('solid', fgColor=(C_ROW_EVEN if ri % 2 == 0 else C_ROW_ODD))
    for ci, (label, _, _) in enumerate(COLS_DETAIL, 1):
        val  = rec.get(label, '')
        cell = ws1.cell(row=ri, column=ci, value=val)
        cell.font      = Font(name='Arial', size=9)
        cell.border    = thin_border()
        cell.alignment = Alignment(vertical='center')
        if 'Status' in label:
            cell.fill = STATUS_FILL.get(str(val), bg)
        elif '% Perubahan' in label:
            cell.fill          = bg
            cell.number_format = '0.0"%"'
        else:
            cell.fill = bg

ws1.freeze_panes = 'A2'
ws1.auto_filter.ref = ws1.dimensions

# ════════════════════════════════════════════════════════════
# SHEET 2 — Ringkasan per NIM
# ════════════════════════════════════════════════════════════
ws2 = wb.create_sheet('Ringkasan per NIM')

COLS_NIM = [
    ('NIM',                          16, 'base'),
    ('Total File',                   12, 'base'),
    ('Black — Berubah',              16, 'black'),
    ('Black — Tidak Berubah',        20, 'black'),
    ('Black — Gagal',                14, 'black'),
    ('Black — Rata-rata % Ubah',     22, 'black'),
    ('Autopep8 — Berubah',           18, 'auto'),
    ('Autopep8 — Tidak Berubah',     22, 'auto'),
    ('Autopep8 — Gagal',             16, 'auto'),
    ('Autopep8 — Rata-rata % Ubah',  24, 'auto'),
]

for ci, (label, width, grp) in enumerate(COLS_NIM, 1):
    c = ws2.cell(row=1, column=ci, value=label)
    c.font      = Font(name='Arial', bold=True, color='FFFFFF', size=10)
    c.fill      = PatternFill('solid', fgColor=GROUP_COLOR[grp])
    c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    c.border    = thin_border()
    ws2.column_dimensions[get_column_letter(ci)].width = width
ws2.row_dimensions[1].height = 35

for ri, (nim, grp) in enumerate(df_report.groupby('NIM'), 2):
    bg = PatternFill('solid', fgColor=(C_ROW_EVEN if ri % 2 == 0 else C_ROW_ODD))
    row_vals = [
        nim,
        len(grp),
        (grp['Black — Status']    == 'Berubah').sum(),
        (grp['Black — Status']    == 'Tidak Berubah').sum(),
        (grp['Black — Status']    == 'Gagal').sum(),
        round(grp['Black — % Perubahan'].mean(), 1),
        (grp['Autopep8 — Status'] == 'Berubah').sum(),
        (grp['Autopep8 — Status'] == 'Tidak Berubah').sum(),
        (grp['Autopep8 — Status'] == 'Gagal').sum(),
        round(grp['Autopep8 — % Perubahan'].mean(), 1),
    ]
    for ci, val in enumerate(row_vals, 1):
        cell = ws2.cell(row=ri, column=ci, value=val)
        cell.font      = Font(name='Arial', size=9)
        cell.fill      = bg
        cell.border    = thin_border()
        cell.alignment = Alignment(horizontal='center', vertical='center')
        if ci in (6, 10):
            cell.number_format = '0.0"%"'

ws2.freeze_panes = 'A2'
ws2.auto_filter.ref = ws2.dimensions

# ════════════════════════════════════════════════════════════
# SHEET 3 — Ringkasan Keseluruhan
# ════════════════════════════════════════════════════════════
ws3 = wb.create_sheet('Ringkasan Keseluruhan')

total_files = len(df_report)
blk_b    = (df_report['Black — Status']    == 'Berubah').sum()
blk_u    = (df_report['Black — Status']    == 'Tidak Berubah').sum()
blk_f    = (df_report['Black — Status']    == 'Gagal').sum()
blk_avg  = round(df_report['Black — % Perubahan'].mean(), 1)
auto_b   = (df_report['Autopep8 — Status'] == 'Berubah').sum()
auto_u   = (df_report['Autopep8 — Status'] == 'Tidak Berubah').sum()
auto_f   = (df_report['Autopep8 — Status'] == 'Gagal').sum()
auto_avg = round(df_report['Autopep8 — % Perubahan'].mean(), 1)

summary_rows = [
    ['METRIK',                        'BLACK',       'AUTOPEP8'     ],
    ['Total File Diproses',            total_files,   total_files    ],
    ['File Berubah',                   blk_b,         auto_b         ],
    ['File Tidak Berubah',             blk_u,         auto_u         ],
    ['File Gagal (syntax error)',      blk_f,         auto_f         ],
    ['Rata-rata % Perubahan',          blk_avg,       auto_avg       ],
    ['',                               '',            ''             ],
    ['KONFIGURASI',                    'BLACK',       'AUTOPEP8'     ],
    ['Line Length Target',             '88 karakter', '79 karakter'  ],
    ['Normalisasi Tanda Kutip String', 'Tidak',       'Tidak'        ],
    ['Perbaiki Indentasi',             'Ya',          'Ya'           ],
    ['Perbaiki Spasi Operator',        'Ya',          'Ya'           ],
    ['Perbaiki Blank Lines',           'Ya',          'Sebagian'     ],
    ['Toleransi Syntax Error',         'Tidak',       'Ya (dilewati)'],
    ['Aggressive Mode',                'Tidak ada',   'Level 1'      ],
]

HDR_ROWS = {1, 8}  # baris yang menjadi header (nomor baris di sheet)

for ri, row in enumerate(summary_rows, 1):
    for ci, val in enumerate(row, 1):
        cell = ws3.cell(row=ri, column=ci, value=val)
        cell.border    = thin_border()
        cell.alignment = Alignment(horizontal='center', vertical='center')
        if ri in HDR_ROWS:
            colors = {1: '2E7D32', 2: C_HDR_BLACK, 3: C_HDR_AUTO}
            cell.font = Font(name='Arial', bold=True, color='FFFFFF', size=11)
            cell.fill = PatternFill('solid', fgColor=colors.get(ci, '333333'))
            ws3.row_dimensions[ri].height = 28
        elif row[0] == '':
            pass  # baris pemisah
        else:
            bg = C_ROW_EVEN if ri % 2 == 0 else C_ROW_ODD
            cell.font = Font(name='Arial', size=10, bold=(ci == 1))
            cell.fill = PatternFill('solid', fgColor=bg)

ws3.column_dimensions['A'].width = 32
ws3.column_dimensions['B'].width = 20
ws3.column_dimensions['C'].width = 20

# ── Simpan file ──────────────────────────────────────────────
wb.save(REPORT_PATH)

print()
print('=' * 70)
print('✅ Laporan Excel berhasil disimpan!')
print(f'   📊 {REPORT_PATH}')
print()
print(f'   📋 Sheet 1 — Detail Per File       : {total_files} baris')
print(f'   📋 Sheet 2 — Ringkasan per NIM     : {df_report["NIM"].nunique()} baris')
print(f'   📋 Sheet 3 — Ringkasan Keseluruhan')
print()
print('  ┌──────────────────────────────┬──────────┬──────────┐')
print('  │ Metrik                       │  Black   │ Autopep8 │')
print('  ├──────────────────────────────┼──────────┼──────────┤')
print(f'  │ File Berubah                 │ {blk_b:>8} │ {auto_b:>8} │')
print(f'  │ File Tidak Berubah           │ {blk_u:>8} │ {auto_u:>8} │')
print(f'  │ File Gagal                   │ {blk_f:>8} │ {auto_f:>8} │')
print(f'  │ Rata-rata % Perubahan        │ {blk_avg:>7}% │ {auto_avg:>7}% │')
print('  └──────────────────────────────┴──────────┴──────────┘')
print('=' * 70)


✅ Laporan Excel berhasil disimpan!
   📊 D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\output(2)\normalisasi_perbandingan_black_autopep.xlsx

   📋 Sheet 1 — Detail Per File       : 2667 baris
   📋 Sheet 2 — Ringkasan per NIM     : 53 baris
   📋 Sheet 3 — Ringkasan Keseluruhan

  ┌──────────────────────────────┬──────────┬──────────┐
  │ Metrik                       │  Black   │ Autopep8 │
  ├──────────────────────────────┼──────────┼──────────┤
  │ File Berubah                 │     2656 │     2365 │
  │ File Tidak Berubah           │       11 │      302 │
  │ File Gagal                   │        0 │        0 │
  │ Rata-rata % Perubahan        │    42.5% │    41.3% │
  └──────────────────────────────┴──────────┴──────────┘


## 6. Filter Submission

In [23]:
import os
import shutil
import time
import pandas as pd
import itertools
from tqdm import tqdm
from openpyxl.styles import PatternFill, Border, Side

# ==============================================================================
# 1. HELPER FUNCTIONS
# ==============================================================================

def scan_directory(base_dir):
    """
    Memindai seluruh folder dataset untuk mendata file .py milik mahasiswa.
    Mengabaikan file yang bukan .py.
    """
    records = []
    if not os.path.exists(base_dir):
        return pd.DataFrame(records)

    for root, _, files in os.walk(base_dir):
        for file in files:
            if file.endswith('.py'):
                rel_path = os.path.relpath(os.path.join(root, file), base_dir)
                parts = rel_path.split(os.sep)
                
                if len(parts) >= 3:
                    nim = parts[0]
                    modul = parts[1]
                    records.append({
                        "nim": nim,
                        "modul": modul,
                        "filename": file,
                        "full_path": os.path.join(root, file) 
                    })
                    
    return pd.DataFrame(records)

def extract_requirements_from_dataset(df_raw, threshold_ratio=0.6):
    """
    Mengekstrak requirement berdasarkan frekuensi kemunculan file.
    Hanya file yang muncul di ≥ threshold mahasiswa yang dianggap requirement.
    """
    req_dict = {}
    if df_raw.empty:
        return req_dict

    for modul in df_raw['modul'].unique():
        df_mod = df_raw[df_raw['modul'] == modul]
        total_students = df_mod['nim'].nunique()

        # Hitung berapa mahasiswa yang punya file tsb
        file_counts = df_mod.groupby('filename_clean')['nim'].nunique()

        # Ambil file yang sering muncul
        valid_files = file_counts[
            file_counts >= threshold_ratio * total_students
        ].index.tolist()

        req_dict[modul] = sorted(valid_files)

    return req_dict

# ==============================================================================
# 2. PROCESSING / TRANSFORM
# ==============================================================================

def create_module_requirements(req_dict):
    """Membangun tabel module_requirements berdasarkan dictionary dinamis."""
    data = []
    for modul in sorted(req_dict.keys()):
        files = req_dict[modul]
        data.append({
            "modul": modul,
            "total_file": len(files),
            "daftar_file": ", ".join(sorted(files))
        })
    return pd.DataFrame(data)

def process_submission_detail(df_raw, req_dict):
    """Memproses detail pengumpulan."""
    nims = sorted(df_raw['nim'].unique()) if not df_raw.empty else []
    modules = list(req_dict.keys())
    
    all_combinations = pd.DataFrame(list(itertools.product(nims, modules)), columns=['nim', 'modul'])
    
    detail_records = []
    for _, row in all_combinations.iterrows():
        nim = row['nim']
        modul = row['modul']
        req_files = set(req_dict.get(modul, []))
        
        if not df_raw.empty:
            submitted = set(df_raw[(df_raw['nim'] == nim) & (df_raw['modul'] == modul)]['filename_clean'])
        else:
            submitted = set()
            
        valid_submitted = submitted.intersection(req_files)
        missing_files = req_files.difference(valid_submitted)
        
        dikerjakan_count = len(valid_submitted)
        target_count = len(req_files)
        persentase = (dikerjakan_count / target_count * 100) if target_count > 0 else 0
        
        status = "Lengkap" if (dikerjakan_count == target_count and target_count > 0) else "Belum Lengkap"
        
        detail_records.append({
            "nim": nim,
            "modul": modul,
            "file_dikerjakan": dikerjakan_count,
            "file_sudah": ", ".join(sorted(valid_submitted)) if valid_submitted else "-",
            "file_kurang": ", ".join(sorted(missing_files)) if missing_files else "-",
            "status": status,
            "target_file": target_count,
            "persentase": round(persentase, 2), 
        })
        
    df_detail = pd.DataFrame(detail_records)
    return df_detail.sort_values(by=['nim', 'modul']).reset_index(drop=True)

def create_submission_matrix(df_detail):
    """Membangun matriks (pivot) status pengerjaan yang lebih informatif."""
    df_detail = df_detail.copy()
    if df_detail.empty:
        return pd.DataFrame()

    # Hitung persentase
    df_detail['persentase'] = (
        df_detail['file_dikerjakan'] / df_detail['target_file']
    ).fillna(0)

    # Format tampilan yang lebih jelas
    def format_cell(row):
        percent = int(row['persentase'] * 100) if row['target_file'] > 0 else 0
        status_icon = "✔️" if row['status'] == "Lengkap" else "❌"
        return f"{row['file_dikerjakan']}/{row['target_file']} ({percent}%) {status_icon}"

    df_detail['matrix_val'] = df_detail.apply(format_cell, axis=1)

    # Pivot matrix
    df_matrix = df_detail.pivot(
        index='nim',
        columns='modul',
        values='matrix_val'
    ).reset_index()

    # Tambahkan total modul selesai
    df_detail['is_lengkap'] = df_detail['status'].apply(lambda x: 1 if x == "Lengkap" else 0)

    modul_lengkap = df_detail.groupby('nim')['is_lengkap'].sum().reset_index()
    modul_lengkap.rename(columns={'is_lengkap': 'total_modul_selesai'}, inplace=True)

    # Tambahkan total modul
    total_modul = df_detail['modul'].nunique()
    
    modul_lengkap['total_modul_tidak_selesai'] = (
        total_modul - modul_lengkap['total_modul_selesai']
    )

    # Hitung progress %
    modul_lengkap['progress (%)'] = (
        modul_lengkap['total_modul_selesai'] / total_modul * 100
    ).round(2)

    df_matrix = pd.merge(df_matrix, modul_lengkap, on='nim', how='left')

    # Urutan kolom lebih rapi
    modul_cols = sorted([c for c in df_matrix.columns if c not in ['nim', 'total_modul_selesai', 'total_modul_tidak_selesai', 'progress (%)']])
    final_cols = ['nim'] + modul_cols + ['total_modul_selesai', 'total_modul_tidak_selesai', 'progress (%)']

    return df_matrix[final_cols]

def create_submission_completeness(df_detail):
    """ Rekap kelengkapan submission per mahasiswa. """
    if df_detail.empty:
        return pd.DataFrame()
    summary_records = []

    for nim, group in df_detail.groupby("nim"):
        # Modul yang memiliki minimal 1 file terkumpul
        modul_dikerjakan = (group["file_dikerjakan"] > 0).sum()
        # Modul lengkap
        modul_lengkap = (group["status"] == "Lengkap").sum()
        # Total file terkumpul
        total_file_terkumpul = group["file_dikerjakan"].sum()
        # Total requirement
        total_requirement = (
            group["file_dikerjakan"].sum()
            + group["file_kurang"].apply(
                lambda x: 0 if x == "-" else len(str(x).split(", "))
            ).sum()
        )
        # Total file missing
        total_file_missing = total_requirement - total_file_terkumpul
        # Persentase kelengkapan
        kelengkapan = (
            total_file_terkumpul / total_requirement * 100
            if total_requirement > 0
            else 0
        )
        summary_records.append({
            "nim": nim,
            "modul_dikerjakan": modul_dikerjakan,
            "modul_lengkap": modul_lengkap,
            "total_file_terkumpul": total_file_terkumpul,
            "total_file_missing": total_file_missing,
            "kelengkapan (%)": round(kelengkapan, 2)
        })
    return pd.DataFrame(summary_records).sort_values("nim")

# ==============================================================================
# 3. EXECUTION (MAIN FUNCTION WITH PHYSICAL FILTER COPY)
# ==============================================================================

def run_tracking_pipeline(base_dir, target_filtered_dir, output_excel):
    print("=" * 70)
    print(f"{' MEMULAI TRACKING & FILTERING SUBMISSION ':^70}")
    print("=" * 70)
    filter_start = start_timer()
    # Reset folder filtered jika sudah ada agar tidak campur baur dengan run lama
    if os.path.exists(target_filtered_dir):
        shutil.rmtree(target_filtered_dir)
    os.makedirs(target_filtered_dir, exist_ok=True)
    
    # 1. Scanning Direktori
    df_raw = scan_directory(base_dir)
    df_raw["filename_clean"] = (
        df_raw["filename"]
        .str.replace(
            r"^MHS\d+_[^_]+_",
            "",
            regex=True
        )
    )
    total_input_modules = df_raw['modul'].nunique()
    total_input_files = len(df_raw)

    # Filter modul yang tidak akan dianalisis
    excluded_modules = {'unclassified', 'kelompok', 'pbl','kuis'}
    df_raw = df_raw[
        ~df_raw['modul'].str.lower().isin(excluded_modules)
    ]
    total_excluded_modules = total_input_modules - df_raw['modul'].nunique()
    
    if df_raw.empty:
        print("⚠️ Direktori kosong atau tidak ditemukan file .py!")
        return None, None, None
        
    # 2. Extract Dynamic Requirements
    threshold_ratio = 0.6
    dynamic_req_dict = extract_requirements_from_dataset(df_raw, threshold_ratio=threshold_ratio)
    modules_without_requirements = [
        modul
        for modul, files in dynamic_req_dict.items()
        if len(files) == 0
    ]
    total_modul = len(dynamic_req_dict)
    total_modules_analyzed = sum(
        1 for files in dynamic_req_dict.values()
        if len(files) > 0
    )
    total_excluded_modules += len(modules_without_requirements)
    print(f"✅ Berhasil mengekstrak requirement dinamis untuk {total_modul} modul. Dengan rasio threshold {threshold_ratio*100:.0f}% yang dikerjakan mahasiswa.")
    analysis_req_dict = {
        modul: files
        for modul, files in dynamic_req_dict.items()
        if len(files) > 0
    }
    total_requirement_files = sum(
        len(files)
        for files in analysis_req_dict.values()
    )
    
    copied_count = 0
    total_loc = 0
    
    # Kita loop baris file mentah hasil scan direktori awal
    for _, row in df_raw.iterrows():        
        nim = row['nim']
        modul = row['modul']
        filename = row['filename']
        src_path = row['full_path']
        filename_clean = row['filename_clean']
        
        # Cek apakah file ini terdaftar sebagai file requirement di modul tersebut
        if filename_clean in dynamic_req_dict.get(modul, []):
            # Tentukan path folder tujuan baru: FILTERED / nim / modul
            dest_folder = os.path.join(target_filtered_dir, nim, modul)
            os.makedirs(dest_folder, exist_ok=True)
            
            # Lakukan penyalinan fisik
            dest_path = os.path.join(dest_folder, filename)
            shutil.copy2(src_path, dest_path)
            copied_count += 1

            file_loc = count_loc(dest_path)
            total_loc += file_loc

    print(f"✅ Pembersihan Sukses! Berhasil memindahkan {copied_count} berkas murni ke: {target_filtered_dir}")
        
    # 3. Processing Data Laporan
    df_req = create_module_requirements(dynamic_req_dict)
    df_detail = process_submission_detail(df_raw, analysis_req_dict)
    df_matrix = create_submission_matrix(df_detail)
    df_completeness = create_submission_completeness(df_detail)
    
    if 'target_file' in df_detail.columns:
        df_detail.drop(columns=['target_file', 'matrix_val', 'is_lengkap'], inplace=True, errors='ignore')

    # 3.5 HITUNG TOTAL FILE REQUIREMENT & SUBMISSION
    total_required_files = sum(len(files) for files in dynamic_req_dict.values())
    total_valid_submitted = df_detail['file_dikerjakan'].sum()
    total_excluded_files = total_input_files - total_valid_submitted
    total_expected_all = total_required_files * df_matrix.shape[0]
    coverage = (total_valid_submitted / total_expected_all * 100) if total_expected_all > 0 else 0
    summary_data = [
        ["Total modul input", total_input_modules],
        ["Total file input", total_input_files],

        ["Total modul excluded", total_excluded_modules],
        ["Total file excluded", total_excluded_files],

        ["Total Modul dianalisis", total_modules_analyzed],
        ["Total file requirement", total_requirement_files],

        ["Total mahasiswa", df_matrix.shape[0]],
        ["Total file seharusnya dikumpulkan", total_expected_all],
        ["Total file berhasil dikumpulkan", total_valid_submitted],

        ["Tingkat kelengkapan submission (%)", round(coverage, 2)],
    ]
        
    df_summary = pd.DataFrame(summary_data, columns=["Metric", "Value"])
    
    # 4. Export ke Excel
    os.makedirs(os.path.dirname(output_excel) if os.path.dirname(output_excel) else '.', exist_ok=True)
    
    # Gunakan format dtype str saat inisiasi all sheets agar NIM tidak hancur menjadi scientific notation
    with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
        if not df_summary.empty:
            df_summary.to_excel(writer, sheet_name='summary', index=False)
        if not df_req.empty:
            df_req.to_excel(writer, sheet_name='module_requirements', index=False)
            worksheet = writer.sheets['module_requirements']
            last_col = df_req.shape[1] + 4
            worksheet.cell(row=1, column=last_col).value = "Keterangan:"
            worksheet.cell(row=2, column=last_col).value = (
                "Sheet ini berisi daftar modul dan file yang harus dikerjakan oleh mahasiswa.\n"
                "Requirement diambil dari {threshold_ratio*100:.0f}% file yang sering muncul pada dataset."
            )
            
            thin = Side(style='thin')
            border = Border(left=thin, right=thin, top=thin, bottom=thin)
            max_row = df_req.shape[0] + 1
            max_col = df_req.shape[1]

            for row in worksheet.iter_rows(min_row=1, max_row=max_row, min_col=1, max_col=max_col):
                for cell in row:
                    cell.border = border
                    
        if not df_detail.empty:
            sheet_name_detail = 'submission_detail'
            df_detail.to_excel(writer, sheet_name=sheet_name_detail, index=False)
            
        if not df_completeness.empty:
            df_completeness.to_excel(writer, sheet_name='submission_completeness', index=False)

        if not df_matrix.empty:
            sheet_name = 'submission_matrix'
            df_matrix.to_excel(writer, sheet_name=sheet_name, index=False)
            worksheet = writer.sheets[sheet_name]

            green_fill = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
            red_fill = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")

            for row in worksheet.iter_rows(min_row=2):
                for cell in row:
                    col_name = df_matrix.columns[cell.column - 1]
                    if col_name in ["nim", "total_modul_selesai", "total_modul_tidak_selesai", "progress (%)"]:
                        continue
                    if cell.value:
                        if "✔️" in str(cell.value): cell.fill = green_fill
                        elif "❌" in str(cell.value): cell.fill = red_fill
            
        # FIX LEBAR KOLOM MINIMAL(AUTO-FIT WIDTH)
        for sheet in writer.sheets:
            worksheet_active = writer.sheets[sheet]
            for column_cells in worksheet_active.columns:
                max_length = max(len(str(cell.value or '')) for cell in column_cells)
                column_letter = column_cells[0].column_letter
                worksheet_active.column_dimensions[column_letter].width = max(max_length + 3, 15)
            
    print(f" Laporan disimpan di            : {output_excel}\n")
    print(f" Total modul terdeteksi         : {total_modul}")
    print(f" Total mahasiswa terdeteksi     : {df_matrix.shape[0]}")
    print("-" * 70)
    print(f"    Jumlah file wajib                         : {total_required_files}")
    print(f"    Total file yang seharusnya dikumpulkan    : {total_expected_all}")
    print(f"    Total file yang berhasil dikumpulkan      : {total_valid_submitted}")
    print(f"    Tingkat kelengkapan submission (%)        : {round(coverage, 2)}%")
    print("-" * 70)
    
    filter_total_time = save_execution_time(filter_start, "6_Filtering", df_matrix.shape[0], total_valid_submitted, get_path_size(target_filtered_dir))
    print(f" Waktu Eksekusi                 : {filter_total_time}")
    print("-" * 70)
    return df_req, df_matrix, df_detail, df_summary

timestamp = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
print("Diproses pada:", timestamp)

Diproses pada: 2026-06-10 05:31:00


In [24]:
# ==============================================================================
# 4. EXECUTION CONTROLLER
# ==============================================================================

# Definisikan koordinat path dari kamus DIRS milikmu
BASE_DIR = DIRS['BLACK']
FILTERED_DIR = DIRS['FILTERED'] # Target folder dataset baru yang bersih total
OUTPUT_EXCEL = RESULTS['SUBMISSION']

# Jalankan pipeline terintegrasi
df_req, df_matrix, df_detail, df_summary = run_tracking_pipeline(BASE_DIR, FILTERED_DIR, OUTPUT_EXCEL)
if df_summary is not None:
    print("[PREVIEW] Sheet 1: summary")
    print("-" * 70)
    display(df_summary)
if df_req is not None:
    # Mengunci preview agar tidak memenuhi layar notebook
    print("[PREVIEW] Sheet 2: module_requirements")
    print("-" * 70)
    display(df_req)

timestamp = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
print("Diproses pada:", timestamp)

               MEMULAI TRACKING & FILTERING SUBMISSION                
✅ Berhasil mengekstrak requirement dinamis untuk 15 modul. Dengan rasio threshold 60% yang dikerjakan mahasiswa.
✅ Pembersihan Sukses! Berhasil memindahkan 2244 berkas murni ke: D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\06_Filtered
 Laporan disimpan di            : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\output(2)\06_submission_report.xlsx

 Total modul terdeteksi         : 15
 Total mahasiswa terdeteksi     : 53
----------------------------------------------------------------------
    Jumlah file wajib                         : 47
    Total file yang seharusnya dikumpulkan    : 2491
    Total file yang berhasil dikumpulkan      : 2244
    Tingkat kelengkapan submission (%)        : 90.08%
----------------------------------------------------------------------

[SELESAI] 6_Filtering | 28.47 s | 2244 file
 Waktu Eksekusi                 : 28.47
--------------------------------------------------------------

,Metric,Value
0,Total modul input,19.00
1,Total file input,2667.00
2,Total modul excluded,7.00
3,Total file excluded,423.00
4,Total Modul dianalisis,12.00
5,Total file requirement,47.00
6,Total mahasiswa,53.00
7,Total file seharusnya dikumpulkan,2491.00
8,Total file berhasil dikumpulkan,2244.00
9,Tingkat kelengkapan submission (%),90.08


[PREVIEW] Sheet 2: module_requirements
----------------------------------------------------------------------


,modul,total_file,daftar_file
0,js01,0,
1,js02,5,"p01.py, p02.py, p03.py, p04.py, tp.py"
2,js03,5,"p01.py, p02.py, p03.py, p04.py, tp.py"
3,js04,4,"p01.py, p02.py, p03.py, tp.py"
4,js05,3,"p01.py, p02.py, tp.py"
5,js06,3,"p01.py, p02.py, tp.py"
6,js07,7,"p01.py, p02.py, p03.py, p04.py, p05.py, p06.py..."
7,js08,1,tp.py
8,js09,5,"p01.py, p02.py, p03.py, tp.py, tp2.py"
9,js11,6,"p01.py, p02.py, p03.py, p04.py, p05.py, tp.py"


Diproses pada: 2026-06-10 05:31:35


## 6a. Split Praktikum & Tugas

In [13]:
# ==============================================================================
# SPLIT DATASET: PRAKTIKUM VS TUGAS
# ==============================================================================
import os
import shutil
import re

def split_practicum_and_assignment(source_dir, practicum_dir, assignment_dir, excel_report):
    """
    Memisahkan dataset ke dalam direktori Praktikum dan Tugas berdasarkan
    pola penamaan file (regex).
    """
    # Bersihkan dan siapkan direktori target
    for target in [practicum_dir, assignment_dir]:
        if os.path.exists(target):
            shutil.rmtree(target)
        os.makedirs(target, exist_ok=True)

    total_practicum, total_assignment = 0, 0

    # Iterasi direktori sumber
    for root, _, files in os.walk(source_dir):
        for file in files:
            if not file.endswith(".py"):
                continue

            source_file = os.path.join(root, file)
            rel_path = os.path.relpath(source_file, source_dir)
            filename = file.lower()

            # Penentuan tujuan berdasarkan pola regex
            if re.search(r"_p\d+\.py$", filename):
                target_file = os.path.join(practicum_dir, rel_path)
                total_practicum += 1
            elif re.search(r"_tp\d*\.py$", filename):
                target_file = os.path.join(assignment_dir, rel_path)
                total_assignment += 1
            else:
                continue

            # Salin berkas ke lokasi baru
            os.makedirs(os.path.dirname(target_file), exist_ok=True)
            shutil.copy2(source_file, target_file)

    # ==============================================================================
    # LOGGING DATASET SPLIT TO EXCEL
    # ==============================================================================
    input_students = count_students(source_dir)
    pract_students = count_students(practicum_dir)
    assign_students = count_students(assignment_dir)
    
    total_files = total_practicum + total_assignment

    summary_split = pd.DataFrame([
        {
            "kategori": "Praktikum",
            "jumlah_mahasiswa": pract_students,
            "jumlah_file": total_practicum,
            "persentase_file": round(
                (total_practicum / total_files * 100),
                2
            ) if total_files > 0 else 0,
            "lokasi_output": practicum_dir,
            "timestamp" : time.strftime("%Y-%m-%d %H:%M:%S")
        },
        {
            "kategori": "Tugas",
            "jumlah_mahasiswa": assign_students,
            "jumlah_file": total_assignment,
            "persentase_file": round(
                (total_assignment / total_files * 100),
                2
            ) if total_files > 0 else 0,
            "lokasi_output": assignment_dir,
            "timestamp" : time.strftime("%Y-%m-%d %H:%M:%S")
        }
    ])

    sheet_name = "splitting_dataset"

    # Konfigurasi mode penulisan Excel secara dinamis
    writer_kwargs = {"engine": "openpyxl", "mode": "w"}
    if os.path.exists(excel_report):
        writer_kwargs.update({"mode": "a", "if_sheet_exists": "replace"})
    with pd.ExcelWriter(excel_report, **writer_kwargs) as writer:
        summary_split.to_excel(writer, sheet_name=sheet_name, index=False)
        
    # Laporan ringkasan eksekusi
    print(f"{'Input direktori':<30}: {source_dir}")
    print(f"{'Total MHS Input':<30}: {input_students}")
    print(f"{'Total File Input':<30}: {count_all_files(source_dir)['total_files']}")
    print(f"{' SELESAI ':-^70}")
    display(summary_split)
    print(f"{'Total File Praktikum':<30}: {total_practicum}")
    print(f"{'Total File Tugas':<30}: {total_assignment}")
    print(f"{'Mahasiswa Praktikum':<30}: {pract_students}")
    print(f"{'Mahasiswa Tugas':<30}: {assign_students}")
    print(f"{'Output Praktikum':<30}: {practicum_dir}")
    print(f"{'Output Tugas':<30}: {assignment_dir}")
    print(f"{'Laporan Excel':<30}: {excel_report} ({sheet_name})")
    print("-" * 70)
    
    return input_students, total_files
    
timestamp = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
print("Diproses pada:", timestamp)

Diproses pada: 2026-06-14 06:20:21


In [14]:
# RUN UNTUK MEMISAHKAN PRAKTIKUM(P) DAN TUGAS(TP)
INPUT_DIR = DIRS['FILTERED']
OUTPUT_P = DIRS['FILTERED_PRAK']
OUTPUT_T = DIRS['FILTERED_TGS']

start = start_timer()
print(f"{' MEMISAHKAN PRAKTIKUM DAN TUGAS ':-^70}")
total_mhs, total_files = split_practicum_and_assignment(
    source_dir=INPUT_DIR,
    practicum_dir=OUTPUT_P,
    assignment_dir=OUTPUT_T,
    excel_report=RESULTS['SUBMISSION']
)

total_size = (get_path_size(OUTPUT_P) + get_path_size(OUTPUT_T))
save_execution_time(start, "6a_Split_Prak_Tugas", total_mhs, total_files, total_size)

timestamp = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
print("Diproses pada:", timestamp)

------------------- MEMISAHKAN PRAKTIKUM DAN TUGAS -------------------
Input direktori               : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\06_Filtered
Total MHS Input               : 53
Total File Input              : 2244
------------------------------ SELESAI -------------------------------


,kategori,jumlah_mahasiswa,jumlah_file,persentase_file,lokasi_output,timestamp
0,Praktikum,53,1678,74.78,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,2026-06-14 06:21:23
1,Tugas,50,566,25.22,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,2026-06-14 06:21:23


Total File Praktikum          : 1678
Total File Tugas              : 566
Mahasiswa Praktikum           : 53
Mahasiswa Tugas               : 50
Output Praktikum              : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\06_Filtered_prak
Output Tugas                  : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\06_Filtered_tgs
Laporan Excel                 : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\output(2)\06_submission_report.xlsx (splitting_dataset)
----------------------------------------------------------------------

[SELESAI] 6a_Split_Prak_Tugas | 65.75 s | 2244 file
Diproses pada: 2026-06-14 06:21:27


## 6b. Pengambilan Sample untuk Ground Truth Modifikasi Terkontrol

In [8]:
# ==============================================================================
# PROPORTIONAL STRATIFIED RANDOM SAMPLING
# ==============================================================================
import random

def proportional_stratified_sampling(source_dir, target_dir, sample_percentage=0.30, report_path=None, random_seed=123, exclude_dirs=None, clean_target=True):
    """
    Melakukan sampling acak berstrata secara proporsional berdasarkan modul.
    """
    if exclude_dirs is None:
        exclude_dirs = []
    random.seed(random_seed)

    if clean_target and os.path.exists(target_dir):
        shutil.rmtree(target_dir)
    os.makedirs(target_dir, exist_ok=True)

    # ------------------------------------------------------------------
    # GROUP FILES BY MODUL
    # ------------------------------------------------------------------
    modul_groups = defaultdict(list)
    for root, _, files in os.walk(source_dir):
        if any(ex in root.split(os.sep) for ex in exclude_dirs):
            continue
        for file in files:
            if not file.endswith(".py"):
                continue
            
            filepath = os.path.join(root, file)
            parts = os.path.relpath(filepath, source_dir).split(os.sep)
            if len(parts) < 3:
                continue

            modul_groups[parts[1]].append({
                "nim": parts[0], "modul": parts[1], 
                "filename": file, "source_path": filepath
            })

    total_pop = sum(len(v) for v in modul_groups.values())
    target_sample_size = round(total_pop * sample_percentage)
    print(f"Source Dir : {source_dir}\nTotal Populasi  : {total_pop}\nTarget Sampel : {target_sample_size} ({sample_percentage * 100} %)\n{'-'*70}")

    # ------------------------------------------------------------------
    # STRATIFIED ALLOCATION
    # ------------------------------------------------------------------
    selected_files, allocation_records = [], []
    for modul, files in sorted(modul_groups.items()):
        n = min(len(files), max(1, round((len(files) / total_pop) * target_sample_size)))
        sampled = random.sample(files, n)
        
        selected_files.extend(sampled)
        allocation_records.append({
            "modul": modul, "total_population": len(files),
            "sample_size": n, "sampling_ratio (%)": round(n / len(files) * 100, 2)
        })
        print(f"{modul:<10} Population={len(files):<5} Sample={n}")

    # ------------------------------------------------------------------
    # CORRECTION & EXPORT
    # ------------------------------------------------------------------
    diff = target_sample_size - len(selected_files)
    if diff > 0:
        used_paths = {x["source_path"] for x in selected_files}
        candidates = [i for files in modul_groups.values() for i in files if i["source_path"] not in used_paths]
        selected_files.extend(random.sample(candidates, min(diff, len(candidates))))
    elif diff < 0:
        selected_files = random.sample(selected_files, target_sample_size)

    sampling_records = []
    for idx, item in enumerate(selected_files, start=1):
        dst = os.path.join(target_dir, item["filename"])
        shutil.copy2(item["source_path"], dst)
        sampling_records.append({"No": idx, **item, "Target_Path": dst})

    df_sample = pd.DataFrame(sampling_records)
    df_summary = pd.DataFrame(allocation_records)

    if report_path:
        os.makedirs(os.path.dirname(report_path), exist_ok=True)
        with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
            df_sample.to_excel(writer, sheet_name="sampled_files", index=False)
            df_summary.to_excel(writer, sheet_name="sampling_summary", index=False)
            for ws in writer.sheets.values():
                for col in ws.columns:
                    max_len = max((len(str(c.value or "")) for c in col), default=0)
                    ws.column_dimensions[col[0].column_letter].width = min(max_len + 3, 60)
                    
        print(f"{'Report':<15}: {report_path}")

    print(f"{'-'*70}\nTotal Sample : {len(df_sample)}\nOutput Dir   : {target_dir}\n{'='*70}")
    return df_sample, df_summary

timestamp = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
print("Diproses pada:", timestamp)

Diproses pada: 2026-06-15 07:48:13


In [9]:
# RUNNING UNTUK PRAKTIKUM
print(f"{'='*70}\nPROPORTIONAL STRATIFIED SAMPLING PRAKTIKUM\n{'='*70}")
INPUT_DIR = DIRS['FILTERED_PRAK']
OUTPUT_DIR = os.path.join(EVAL_P['SAMPLE'], 'asli')
REPORT_PATH = RESULTS_EVAL_P['SAMPLING_REPORT']

start = start_timer()
df_sample, df_summary = proportional_stratified_sampling(
    source_dir=INPUT_DIR,
    target_dir=OUTPUT_DIR,
    sample_percentage=0.30,
    report_path=REPORT_PATH
)

waktu_eksekusi = save_execution_time(start, "6b_Sampling_Praktikum", count_students(INPUT_DIR), count_all_files(OUTPUT_DIR)['total_files'], get_path_size(OUTPUT_DIR))

display(df_sample.head())
display(df_summary.head())

PROPORTIONAL STRATIFIED SAMPLING PRAKTIKUM
Source Dir : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\06_Filtered_prak
Total Populasi  : 1678
Target Sampel : 503 (30.0 %)
----------------------------------------------------------------------
js02       Population=199   Sample=60
js03       Population=197   Sample=59
js04       Population=141   Sample=42
js05       Population=95    Sample=28
js06       Population=98    Sample=29
js07       Population=277   Sample=83
js09       Population=146   Sample=44
js11       Population=245   Sample=73
js13       Population=145   Sample=43
js14       Population=91    Sample=27
js15       Population=44    Sample=13
Report         : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\output(2)\eval\praktikum\06_sampling_report.xlsx
----------------------------------------------------------------------
Total Sample : 503
Output Dir   : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\praktikum\06_SAMPLE\asli

[SELESAI] 6b_Sampling_Praktikum | 9.0 s

,No,nim,modul,filename,source_path,Target_Path
0,1,MHS004,js02,MHS004_js02_p02.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...
1,2,MHS018,js02,MHS018_js02_p01.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...
2,3,MHS006,js02,MHS006_js02_p03.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...
3,4,MHS028,js02,MHS028_js02_p03.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...
4,5,MHS053,js02,MHS053_js02_p03.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...


,modul,total_population,sample_size,sampling_ratio (%)
0,js02,199,60,30.15
1,js03,197,59,29.95
2,js04,141,42,29.79
3,js05,95,28,29.47
4,js06,98,29,29.59


In [10]:
# RUNNING UNTUK PRAKTIKUM
print(f"{'='*70}\nPROPORTIONAL STRATIFIED SAMPLING TUGAS\n{'='*70}")
INPUT_DIR = DIRS['FILTERED_TGS']
OUTPUT_DIR = os.path.join(EVAL_T['SAMPLE'], 'asli')
REPORT_PATH = RESULTS_EVAL_T['SAMPLING_REPORT']

start = start_timer()
df_sample, df_summary = proportional_stratified_sampling(
    source_dir=INPUT_DIR,
    target_dir=OUTPUT_DIR,
    sample_percentage=0.30,
    report_path=REPORT_PATH
)

waktu_eksekusi = save_execution_time(start, "6b_Sampling_Tugas", count_students(INPUT_DIR), count_all_files(OUTPUT_DIR)['total_files'], get_path_size(OUTPUT_DIR))

display(df_sample.head())
display(df_summary.head())

PROPORTIONAL STRATIFIED SAMPLING TUGAS
Source Dir : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\06_Filtered_tgs
Total Populasi  : 566
Target Sampel : 170 (30.0 %)
----------------------------------------------------------------------
js02       Population=49    Sample=15
js03       Population=49    Sample=15
js04       Population=45    Sample=14
js05       Population=48    Sample=14
js06       Population=46    Sample=14
js07       Population=44    Sample=13
js08       Population=47    Sample=14
js09       Population=98    Sample=29
js11       Population=47    Sample=14
js13       Population=49    Sample=15
js14       Population=44    Sample=13
Report         : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\output(2)\eval\tugas\06_sampling_report.xlsx
----------------------------------------------------------------------
Total Sample : 170
Output Dir   : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\tugas\06_SAMPLE\asli

[SELESAI] 6b_Sampling_Tugas | 3.27 s | 170 file


,No,nim,modul,filename,source_path,Target_Path
0,1,MHS004,js02,MHS004_js02_tp.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...
1,2,MHS018,js02,MHS018_js02_tp.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...
2,3,MHS006,js02,MHS006_js02_tp.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...
3,4,MHS029,js02,MHS029_js02_tp.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...
4,5,MHS052,js02,MHS052_js02_tp.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...


,modul,total_population,sample_size,sampling_ratio (%)
0,js02,49,15,30.61
1,js03,49,15,30.61
2,js04,45,14,31.11
3,js05,48,14,29.17
4,js06,46,14,30.43


## 6c. Generate/Create Klon 

In [8]:
import builtins
from openpyxl import load_workbook
import black
# ==============================================================================
# HELPER ENGINE: GENERATE VARIATION TYPE-1
# ==============================================================================
def create_type1_clone(source_file, target_file):
    """
    Klon Kode Tipe-1 (Exact Clone dengan modifikasi teks minor):
    - Menyisipkan komentar header otomatis.
    - Menghapus komentar bawaan mahasiswa agar standarisasi terpenuhi.
    - Menambahkan baris kosong (blank line) acak tanpa merubah struktur AST.
    """
    with open(source_file, "r", encoding="utf-8") as f:
        lines = f.readlines()

    new_lines = [
        "# ======================================\n",
        "# Clone Type-1 Generated Automatically\n",
        "# ======================================\n\n"
    ]

    for line in lines:
        stripped = line.strip()
        # Eksklusi komentar lama untuk menguji ketahanan model kemiripan kode
        if stripped.startswith("#"):
            continue

        new_lines.append(line)

        # Injeksi blank line acak dengan probabilitas 10%
        if random.random() < 0.10:
            new_lines.append("\n")

    with open(target_file, "w", encoding="utf-8") as f:
        f.writelines(new_lines)

# 2. HELPER NODE TRANSFORMER: ENGINE GANTI NAMA VARIABEL & FUNGSI (CLONE TYPE 2)
class RefactorCloneType2AST(ast.NodeTransformer):
    def __init__(self, tree):
        super().__init__()
        # mapping nama lama -> nama baru
        self.variable_map = {}
        self.function_map = {}

        # nama library/import
        self.imported_names = set()
        # builtin python
        self.builtin_names = set(dir(builtins))
        # whitelist argumen standar agar tidak merusak fungsi bawaan (misal: print(..., end=" "))
        self.common_kwargs_whitelist = {'end', 'sep', 'file', 'flush', 'key', 'reverse', 'mode', 'encoding', 'errors', 'inplace'}
        # Jalankan pemindaian awal untuk mengumpulkan metadata secara global sebelum mutasi
        self._pre_scan_metadata(tree)

    def _pre_scan_metadata(self, tree):
        """Memindai seluruh dokumen terlebih dahulu untuk menghindari bug urutan baris."""
        for node in ast.walk(tree):
            if isinstance(node, ast.Import):
                for alias in node.names:
                    self.imported_names.add(alias.asname or alias.name.split(".")[0])
            elif isinstance(node, ast.ImportFrom):
                if node.module:
                    self.imported_names.add(node.module.split(".")[0])
                for alias in node.names:
                    self.imported_names.add(alias.asname or alias.name)
            elif isinstance(node, ast.FunctionDef):
                if node.name not in self.builtin_names:
                    self.function_map[node.name] = f"plagiat_func_{node.name}"

    # FUNCTION DEF
    def visit_FunctionDef(self, node):
        old_name = node.name
        if old_name in self.function_map:
            node.name = self.function_map[old_name]
        # rename parameter
        if node.args and node.args.args:
            for arg in node.args.args:
                old_arg = arg.arg
                if old_arg not in self.builtin_names and old_arg not in self.imported_names:
                    if old_arg not in self.variable_map:
                        self.variable_map[old_arg] = f"plagiat_var_{old_arg}"
                    arg.arg = self.variable_map[old_arg]
        self.generic_visit(node)
        return node

    # LAMBDA EXPRESSION (Kasus Lambda Arguments)
    def visit_Lambda(self, node):
        # Rename parameter internal milik lambda agar sinkron dengan bodynya
        if node.args and node.args.args:
            for arg in node.args.args:
                old_arg = arg.arg
                if old_arg not in self.builtin_names and old_arg not in self.imported_names:
                    if old_arg not in self.variable_map:
                        self.variable_map[old_arg] = f"plagiat_var_{old_arg}"
                    arg.arg = self.variable_map[old_arg]
        self.generic_visit(node)
        return node
    
    # FOR LOOP VARIABLE
    def visit_For(self, node):
        # Tangani jika target berupa single variable (for i in ...)
        if isinstance(node.target, ast.Name):
            old_name = node.target.id
            if old_name not in self.builtin_names and old_name not in self.imported_names:
                if old_name not in self.variable_map:
                    self.variable_map[old_name] = f"plagiat_var_type2_{old_name}"
        # Tangani jika target berupa tuple unpacking (for idx, row in ...)
        elif isinstance(node.target, ast.Tuple):
            for elt in node.target.elts:
                if isinstance(elt, ast.Name):
                    old_name = elt.id
                    if old_name not in self.builtin_names and old_name not in self.imported_names:
                        if old_name not in self.variable_map:
                            self.variable_map[old_name] = f"plagiat_var_type2_{old_name}"
        self.generic_visit(node)
        return node

    # VARIABLE
    def visit_Name(self, node):
        name = node.id
        # jangan rename builtin
        if name in self.builtin_names:
            return node
        # jangan rename library/import
        if name in self.imported_names:
            return node
        # rename deklarasi variabel
        if isinstance(node.ctx, ast.Store):
            if name in self.function_map:
                node.id = self.function_map[name]
            else:
                if name not in self.variable_map:
                    self.variable_map[name] = f"plagiat_var_type2_{name}"
                node.id = self.variable_map[name]
        # rename penggunaan variabel
        elif isinstance(node.ctx, ast.Load):
            if name in self.variable_map:
                node.id = self.variable_map[name]
            elif name in self.function_map:
                node.id = self.function_map[name]
            else:
                # Fallback untuk variabel global/kondisional yang tidak melewati ast.Store di awal
                self.variable_map[name] = f"plagiat_var_type2_{name}"
                node.id = self.variable_map[name]
        return node

    # FUNCTION CALL & KEYWORDS
    def visit_Call(self, node):
        # Deteksi awal: Apakah sedang memanggil fungsi kustom buatan mahasiswa?
        is_user_func = isinstance(node.func, ast.Name) and node.func.id in self.function_map

        # Kunjungi fungsi pemanggil dan argumen posisionalnya
        node.func = self.visit(node.func)
        node.args = [self.visit(arg) for arg in node.args]
        # Tangani keyword arguments secara dinamis
        new_keywords = []
        for kw in node.keywords:
            if kw.arg:
                if is_user_func and kw.arg not in self.common_kwargs_whitelist:
                    if kw.arg in self.variable_map:
                        kw.arg = self.variable_map[kw.arg]
                    else:
                        new_kw_name = f"plagiat_var_type2_{kw.arg}"
                        self.variable_map[kw.arg] = new_kw_name
                        kw.arg = new_kw_name
            if kw.value:
                kw.value = self.visit(kw.value)
            new_keywords.append(kw)
        node.keywords = new_keywords
        return node

# HELPER TYPE
class CloneType3AST(ast.NodeTransformer):
    def __init__(self):
        super().__init__()

    # MODULE LEVEL TRANSFORMATIONS
    def visit_Module(self, node):
        self.generic_visit(node)
        helper_1 = ast.FunctionDef(
            name="__clone_identity_1",
            args=ast.arguments(
                posonlyargs=[],
                args=[ast.arg(arg="x")],
                kwonlyargs=[],
                kw_defaults=[],
                defaults=[]
            ),
            body=[
                ast.Return(
                    value=ast.Name(
                        id="x",
                        ctx=ast.Load()
                    )
                )
            ],
            decorator_list=[]
        )
        node.body.insert(0, helper_1)
        mutation_count = random.randint(5, 10)
        for _ in range(mutation_count):
            stmt = ast.Assign(
                targets=[
                    ast.Name(
                        id=f"__clone_global_{random.randint(1,9999)}",
                        ctx=ast.Store()
                    )
                ],
                value=ast.Constant(
                    random.randint(0,1000)
                )
            )
            node.body.append(stmt)
        return node
    
    def visit_For(self, node):
        self.generic_visit(node)
        if random.random() > 0.30:
            return node
        node.iter = ast.Call(
            func=ast.Name(
                id="__clone_identity_1",
                ctx=ast.Load()
            ),
            args=[node.iter],
            keywords=[]
        )
        return node
    
    def visit_Call(self, node):
        self.generic_visit(node)
        if random.random() > 0.25:
            return node
        return ast.Call(
            func=ast.Name(
                id="__clone_identity_1",
                ctx=ast.Load()
            ),
            args=[node],
            keywords=[]
        )
    
    def visit_Return(self, node):
        self.generic_visit(node)
        if node.value is None:
            return node
        return ast.Return(
            value=ast.Call(
                func=ast.Name(
                    id="__clone_identity_1",
                    ctx=ast.Load()
                ),
                args=[node.value],
                keywords=[]
            )
        )
        
    # FUNCTION LEVEL TRANSFORMATIONS
    def visit_FunctionDef(self, node):
        self.generic_visit(node)
        if not node.body:
            return node
        insert_count = random.randint(5, 10)
        for _ in range(insert_count):
            mutation = random.choice([
                "assign",
                "expr",
                "if",
                "loop",
                "try"
            ])
            # ASSIGNMENT
            if mutation == "assign":
                new_stmt = ast.Assign(
                    targets=[
                        ast.Name(
                            id=f"__clone_temp_{random.randint(1,9999)}",
                            ctx=ast.Store()
                        )
                    ],
                    value=ast.Constant(
                        random.randint(0,1000)
                    )
                )
            # STRING EXPRESSION
            elif mutation == "expr":
                new_stmt = ast.Expr(
                    value=ast.Constant(
                        value=f"clone_marker_{random.randint(1,9999)}"
                    )
                )
            # DUMMY IF
            elif mutation == "if":
                new_stmt = ast.If(
                    test=ast.Constant(True),
                    body=[
                        ast.Assign(
                            targets=[
                                ast.Name(
                                    id=f"__clone_flag_{random.randint(1,9999)}",
                                    ctx=ast.Store()
                                )
                            ],
                            value=ast.Constant(1)
                        )
                    ],
                    orelse=[]
                )
            # DUMMY LOOP
            elif mutation == "loop":
                new_stmt = ast.For(
                    target=ast.Name(
                        id=f"__clone_i_{random.randint(1,9999)}",
                        ctx=ast.Store()
                    ),
                    iter=ast.Call(
                        func=ast.Name(
                            id="range",
                            ctx=ast.Load()
                        ),
                        args=[
                            ast.Constant(
                                random.randint(1,3)
                            )
                        ],
                        keywords=[]
                    ),
                    body=[
                        ast.Pass()
                    ],
                    orelse=[]
                )
            # DUMMY TRY EXCEPT
            else:
                new_stmt = ast.Try(
                    body=[
                        ast.Pass()
                    ],
                    handlers=[
                        ast.ExceptHandler(
                            type=ast.Name(
                                id="Exception",
                                ctx=ast.Load()
                            ),
                            name=None,
                            body=[
                                ast.Pass()
                            ]
                        )
                    ],
                    orelse=[],
                    finalbody=[]
                )
            valid_positions = []
            for idx, stmt in enumerate(node.body):
                if not isinstance(
                    stmt,
                    (
                        ast.Return,
                        ast.Raise,
                        ast.Break,
                        ast.Continue
                    )
                ):
                    valid_positions.append(idx)
            posisi = (
                random.choice(valid_positions)
                if valid_positions
                else 0
            )
            node.body.insert(
                posisi,
                new_stmt
            )

        return node

    def visit_Assign(self, node):
        self.generic_visit(node)
        # hanya assignment tunggal
        if len(node.targets) != 1:
            return node
        # hanya untuk operasi biner
        if not isinstance(node.value, ast.BinOp):
            return node
        # hanya sebagian assignment agar tidak terlalu agresif
        if random.random() > 0.50:
            return node
        left_var = ast.Name(
            id=f"__clone_left_{random.randint(1,9999)}",
            ctx=ast.Store()
        )
        right_var = ast.Name(
            id=f"__clone_right_{random.randint(1,9999)}",
            ctx=ast.Store()
        )
        left_assign = ast.Assign(
            targets=[left_var],
            value=node.value.left
        )
        right_assign = ast.Assign(
            targets=[right_var],
            value=node.value.right
        )
        new_assign = ast.Assign(
            targets=node.targets,
            value=ast.BinOp(
                left=ast.Name(
                    id=left_var.id,
                    ctx=ast.Load()
                ),
                op=node.value.op,
                right=ast.Name(
                    id=right_var.id,
                    ctx=ast.Load()
                )
            )
        )

        return [
            left_assign,
            right_assign,
            new_assign
        ]
        
    def visit_AsyncFunctionDef(self, node):
        self.generic_visit(node)
        if not node.body:
            return node
        # Konstruksi subtree asinkron dummy penanda modifikasi kontrol biner
        dummy_stmt = ast.Assign(
            targets=[
                ast.Name(id=f"__clone_async_{random.randint(1, 9999)}", ctx=ast.Store())
            ],
            value=ast.Constant(random.randint(0, 100))
        )

        node.body.insert(0, dummy_stmt)
        return node
    
    def visit_Compare(self, node):
        self.generic_visit(node)
        if random.random() > 0.25:
            return node
        if len(node.ops) != 1:
            return node

        left_name = f"__clone_cmp_left_{random.randint(1,9999)}"
        right_name = f"__clone_cmp_right_{random.randint(1,9999)}"

        return ast.Compare(
            left=ast.Call(
                func=ast.Name(
                    id="__clone_identity_1",
                    ctx=ast.Load()
                ),
                args=[node.left],
                keywords=[]
            ),
            ops=node.ops,
            comparators=[
                ast.Call(
                    func=ast.Name(
                        id="__clone_identity_1",
                        ctx=ast.Load()
                    ),
                    args=[node.comparators[0]],
                    keywords=[]
                )
            ]
        )

class CloneType4AST(ast.NodeTransformer):
    def __init__(self):
        super().__init__()
        self.wrapper_functions = []
        self.wrapper_count = 0
        self.helper_created = False

    def visit_Module(self, node):
        self.generic_visit(node)
        helper_1 = ast.FunctionDef(
            name="__clone_chain_1",
            args=ast.arguments(
                posonlyargs=[],
                args=[ast.arg(arg="x")],
                kwonlyargs=[],
                kw_defaults=[],
                defaults=[]
            ),
            body=[
                ast.Return(
                    value=ast.Name(
                        id="x",
                        ctx=ast.Load()
                    )
                )
            ],
            decorator_list=[]
        )

        helper_2 = ast.FunctionDef(
            name="__clone_chain_2",
            args=ast.arguments(
                posonlyargs=[],
                args=[ast.arg(arg="x")],
                kwonlyargs=[],
                kw_defaults=[],
                defaults=[]
            ),
            body=[
                ast.Return(
                    value=ast.Call(
                        func=ast.Name(
                            id="__clone_chain_1",
                            ctx=ast.Load()
                        ),
                        args=[
                            ast.Name(
                                id="x",
                                ctx=ast.Load()
                            )
                        ],
                        keywords=[]
                    )
                )
            ],
            decorator_list=[]
        )
        original_body = node.body

        inner_wrapper = ast.FunctionDef(
            name=f"__clone_wrapper_inner_{random.randint(1,9999)}",
            args=ast.arguments(
                posonlyargs=[],
                args=[],
                kwonlyargs=[],
                kw_defaults=[],
                defaults=[]
            ),
            body=original_body,
            decorator_list=[]
        )
        outer_wrapper = ast.FunctionDef(
            name=f"__clone_wrapper_outer_{random.randint(1,9999)}",
            args=ast.arguments(
                posonlyargs=[],
                args=[],
                kwonlyargs=[],
                kw_defaults=[],
                defaults=[]
            ),
            body=[
                inner_wrapper,
                ast.Expr(
                    value=ast.Call(
                        func=ast.Name(
                            id=inner_wrapper.name,
                            ctx=ast.Load()
                        ),
                        args=[],
                        keywords=[]
                    )
                )
            ],
            decorator_list=[]
        )
        node.body = (
            [helper_1, helper_2]
            + self.wrapper_functions
            + [
                outer_wrapper,
                ast.Expr(
                    value=ast.Call(
                        func=ast.Name(
                            id=outer_wrapper.name,
                            ctx=ast.Load()
                        ),
                        args=[],
                        keywords=[]
                    )
                )
            ]
        )
        return node
    
    @staticmethod
    def get_loaded_names(node):
        """
        Mengambil seluruh variabel yang digunakan (Load context)
        pada sebuah expression.
        """
        names = []
        for child in ast.walk(node):
            if (
                isinstance(child, ast.Name)
                and isinstance(child.ctx, ast.Load)
            ):
                names.append(child.id)
        return sorted(set(names))
    
    def visit_Assign(self, node):        
        self.generic_visit(node)
        # hanya sebagian assignment
        if random.random() > 0.30:
            return node
        # hanya assignment tunggal
        if len(node.targets) != 1:
            return node
        target = node.targets[0]
        # hanya target berupa variable
        if not isinstance(target, ast.Name):
            return node

        if not isinstance(
            node.value,
            (
                ast.BinOp,
                ast.Call,
                ast.Compare
            )
        ):
            return node

        helper_name = (
            f"__clone_extract_{random.randint(1000,9999)}"
        )
        # cari dependency variable
        input_names = self.get_loaded_names(node.value)
        helper_args = [
            ast.arg(arg=name)
            for name in input_names
        ]
        helper_func = ast.FunctionDef(
            name=helper_name,
            args=ast.arguments(
                posonlyargs=[],
                args=helper_args,
                kwonlyargs=[],
                kw_defaults=[],
                defaults=[]
            ),
            body=[
                ast.Return(
                    value=node.value
                )
            ],
            decorator_list=[]
        )
        self.wrapper_functions.append(
            helper_func
        )
        return ast.Assign(
            targets=node.targets,
            value=ast.Call(
                func=ast.Name(
                    id=helper_name,
                    ctx=ast.Load()
                ),
                args=[
                    ast.Name(
                        id=name,
                        ctx=ast.Load()
                    )
                    for name in input_names
                ],
                keywords=[]
            )
        )
    
    def visit_Return(self, node):
        self.generic_visit(node)
        if node.value is None:
            return node
        return ast.Return(
            value=ast.Call(
                func=ast.Name(
                    id="__clone_chain_2",
                    ctx=ast.Load()
                ),
                args=[
                    ast.Call(
                        func=ast.Name(
                            id="__clone_chain_1",
                            ctx=ast.Load()
                        ),
                        args=[node.value],
                        keywords=[]
                    )
                ],
                keywords=[]
            )
        )
        
    def visit_Compare(self, node):
        self.generic_visit(node)
        if random.random() > 0.50:
            return node

        return ast.Call(
            func=ast.Name(
                id="bool",
                ctx=ast.Load()
            ),
            args=[
                ast.Call(
                    func=ast.Name(
                        id="__clone_chain_1",
                        ctx=ast.Load()
                    ),
                    args=[node],
                    keywords=[]
                )
            ],
            keywords=[]
        )
    

    def visit_Call(self, node):
        self.generic_visit(node)
        if random.random() > 0.70:
            return node

        return ast.Call(
            func=ast.Name(
                id="__clone_chain_2",
                ctx=ast.Load()
            ),
            args=[
                ast.Call(
                    func=ast.Name(
                        id="__clone_chain_1",
                        ctx=ast.Load()
                    ),
                    args=[node],
                    keywords=[]
                )
            ],
            keywords=[]
        )
        
    def visit_For(self, node):
        self.generic_visit(node)
        if random.random() > 0.70:
            return node
        node.iter = ast.Call(
            func=ast.Name(
                id="__clone_chain_2",
                ctx=ast.Load()
            ),
            args=[node.iter],
            keywords=[]
        )

        return node

# ==============================================================================
# HELPER - TYPE 2 ENGINE
# ==============================================================================

def create_type2_clone(source_file, target_file):
    with open(source_file, "r", encoding="utf-8") as f:
        source_code = f.read()
    tree = ast.parse(source_code)
    transformer = RefactorCloneType2AST(tree)
    tree = transformer.visit(tree)
    ast.fix_missing_locations(tree)
    result = ast.unparse(tree)
    result = black.format_str(
        result,
        mode=black.FileMode()
    )
    ast.parse(result)
    with open(target_file, "w", encoding="utf-8") as f:
        f.write(result)


# ==============================================================================
# HELPER - TYPE 3 ENGINE
# ==============================================================================
def create_type3_clone(source_file, target_file):
    with open(source_file, "r", encoding="utf-8") as f:
        source_code = f.read()

    tree = ast.parse(source_code)
    transformer_type3 = CloneType3AST()
    tree = transformer_type3.visit(tree)
    ast.fix_missing_locations(tree)
    result = ast.unparse(tree)
    result = black.format_str(
        result,
        mode=black.FileMode()
    )
    ast.parse(result)
    with open(target_file, "w", encoding="utf-8") as f:
        f.write(result)

# ==============================================================================
# HELPER - TYPE 4 ENGINE
# ==============================================================================
def create_type4_clone(source_file, target_file):
    with open(source_file, "r", encoding="utf-8") as f:
        source_code = f.read()
    tree = ast.parse(source_code)
    transformer_type4 = CloneType4AST()
    tree = transformer_type4.visit(tree)
    ast.fix_missing_locations(tree)
    result = ast.unparse(tree)
    # validasi syntax
    ast.parse(result)
    result = black.format_str(
        result,
        mode=black.FileMode()
    )

    with open(target_file, "w", encoding="utf-8") as f:
        f.write(result)

# ==============================================================================
# EXPORT HELPER
# ==============================================================================

def export_clone_log(df_log, excel_path, sheet_name):
    mode = "a" if os.path.exists(excel_path) else "w"
    with pd.ExcelWriter(excel_path, engine="openpyxl", mode=mode, if_sheet_exists="replace" if mode == "a" else None) as writer:
        df_log.to_excel(
            writer,
            sheet_name=sheet_name,
            index=False
        )

        ws = writer.sheets[sheet_name]
        for col in ws.columns:
            max_len = max(
                len(str(cell.value or ""))
                for cell in col
            )
            ws.column_dimensions[
                col[0].column_letter
            ].width = min(max_len + 3, 60)


# ==============================================================================
# UNIVERSAL CLONE GENERATOR
# ==============================================================================
def generate_clone_dataset(clone_type, sample_report_path, target_dir):
    if os.path.exists(target_dir):
        shutil.rmtree(target_dir)
    os.makedirs(target_dir, exist_ok=True)
    df_master = pd.read_excel(
        sample_report_path,
        sheet_name="sampled_files"
    )
    logs = []
    random.seed(42)
    print("=" * 70)
    print(f"GENERATE {clone_type.upper()}")
    print("=" * 70)

    for idx, row in df_master.iterrows():
        source_file = row["Target_Path"]
        filename = row["filename"]
        output_name = filename.replace(
            ".py",
            f"_{clone_type}.py"
        )
        target_file = os.path.join(
            target_dir,
            output_name
        )
        try:
            if clone_type == "type1":
                create_type1_clone(
                    source_file,
                    target_file
                )
            elif clone_type == "type2":
                create_type2_clone(
                    source_file,
                    target_file
                )
            elif clone_type == "type3":
                create_type3_clone(
                    source_file,
                    target_file
                )
            elif clone_type == "type4":
                create_type4_clone(
                    source_file,
                    target_file
                )
                
            status = "SUCCESS"
            note = ""

        except Exception as e:
            status = "FAILED"
            note = str(e)

        logs.append({
            "No": idx + 1,
            "NIM": row["nim"],
            "Modul": row["modul"],
            "Clone_Type": clone_type,
            "Nama_File_Asli": filename,
            "Nama_File_Clone": output_name,
            "Path_Asli": source_file,
            "Path_Clone": target_file,
            "Status": status,
            "Keterangan": note
        })

    df_log = pd.DataFrame(logs)

    success_count = len(
        df_log[df_log["Status"] == "SUCCESS"]
    )

    print(f"Total File  : {len(df_log)}")
    print(f"Success     : {success_count}")
    print(f"Failed      : {len(df_log) - success_count}")
    print(f"Output      : {target_dir}")

    return df_log

print(f"Diproses pada {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Diproses pada 2026-06-17 02:56:38


In [9]:
# ==============================================================================
# GENERATE TYPE 1 - TYPE 4 (PRAKTIKUM)
# ==============================================================================
print(f"{'GENERATE KLON PRAKTIKUM (TYPE 1 - 4)':^50}")
EVAL_TYPE1_DIR = os.path.join(EVAL_P["SAMPLE"], "type1")
EVAL_TYPE2_DIR = os.path.join(EVAL_P["SAMPLE"], "type2")
EVAL_TYPE3_DIR = os.path.join(EVAL_P["SAMPLE"], "type3")
EVAL_TYPE4_DIR = os.path.join(EVAL_P["SAMPLE"], "type4")
REPORT_PATH = RESULTS_EVAL_P["SAMPLING_REPORT"]

df_type1 = generate_clone_dataset(
    clone_type="type1",
    sample_report_path=REPORT_PATH,
    target_dir=EVAL_TYPE1_DIR
)

df_type2 = generate_clone_dataset(
    clone_type="type2",
    sample_report_path=REPORT_PATH,
    target_dir=EVAL_TYPE2_DIR
)

df_type3 = generate_clone_dataset(
    clone_type="type3",
    sample_report_path=REPORT_PATH,
    target_dir=EVAL_TYPE3_DIR
)

df_type4 = generate_clone_dataset(
    clone_type="type4",
    sample_report_path=REPORT_PATH,
    target_dir=EVAL_TYPE4_DIR
)

export_clone_log(
    df_type1,
    REPORT_PATH,
    "Type1_Code"
)

export_clone_log(
    df_type2,
    REPORT_PATH,
    "Type2_Code"
)

export_clone_log(
    df_type3,
    REPORT_PATH,
    "Type3_Code"
)

export_clone_log(
    df_type4,
    REPORT_PATH,
    "Type4_Code"
)

display(df_type1.head())
display(df_type2.head())
display(df_type3.head())
display(df_type4.head())
print(f"Diproses pada {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

       GENERATE KLON PRAKTIKUM (TYPE 1 - 4)       
GENERATE TYPE1
Total File  : 503
Success     : 503
Failed      : 0
Output      : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\praktikum\06_SAMPLE\type1
GENERATE TYPE2
Total File  : 503
Success     : 503
Failed      : 0
Output      : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\praktikum\06_SAMPLE\type2
GENERATE TYPE3
Total File  : 503
Success     : 503
Failed      : 0
Output      : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\praktikum\06_SAMPLE\type3
GENERATE TYPE4
Total File  : 503
Success     : 503
Failed      : 0
Output      : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\praktikum\06_SAMPLE\type4


,No,NIM,Modul,Clone_Type,Nama_File_Asli,Nama_File_Clone,Path_Asli,Path_Clone,Status,Keterangan
0,1,MHS004,js02,type1,MHS004_js02_p02.py,MHS004_js02_p02_type1.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,
1,2,MHS018,js02,type1,MHS018_js02_p01.py,MHS018_js02_p01_type1.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,
2,3,MHS006,js02,type1,MHS006_js02_p03.py,MHS006_js02_p03_type1.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,
3,4,MHS028,js02,type1,MHS028_js02_p03.py,MHS028_js02_p03_type1.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,
4,5,MHS053,js02,type1,MHS053_js02_p03.py,MHS053_js02_p03_type1.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,


,No,NIM,Modul,Clone_Type,Nama_File_Asli,Nama_File_Clone,Path_Asli,Path_Clone,Status,Keterangan
0,1,MHS004,js02,type2,MHS004_js02_p02.py,MHS004_js02_p02_type2.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,
1,2,MHS018,js02,type2,MHS018_js02_p01.py,MHS018_js02_p01_type2.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,
2,3,MHS006,js02,type2,MHS006_js02_p03.py,MHS006_js02_p03_type2.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,
3,4,MHS028,js02,type2,MHS028_js02_p03.py,MHS028_js02_p03_type2.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,
4,5,MHS053,js02,type2,MHS053_js02_p03.py,MHS053_js02_p03_type2.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,


,No,NIM,Modul,Clone_Type,Nama_File_Asli,Nama_File_Clone,Path_Asli,Path_Clone,Status,Keterangan
0,1,MHS004,js02,type3,MHS004_js02_p02.py,MHS004_js02_p02_type3.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,
1,2,MHS018,js02,type3,MHS018_js02_p01.py,MHS018_js02_p01_type3.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,
2,3,MHS006,js02,type3,MHS006_js02_p03.py,MHS006_js02_p03_type3.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,
3,4,MHS028,js02,type3,MHS028_js02_p03.py,MHS028_js02_p03_type3.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,
4,5,MHS053,js02,type3,MHS053_js02_p03.py,MHS053_js02_p03_type3.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,


,No,NIM,Modul,Clone_Type,Nama_File_Asli,Nama_File_Clone,Path_Asli,Path_Clone,Status,Keterangan
0,1,MHS004,js02,type4,MHS004_js02_p02.py,MHS004_js02_p02_type4.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,
1,2,MHS018,js02,type4,MHS018_js02_p01.py,MHS018_js02_p01_type4.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,
2,3,MHS006,js02,type4,MHS006_js02_p03.py,MHS006_js02_p03_type4.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,
3,4,MHS028,js02,type4,MHS028_js02_p03.py,MHS028_js02_p03_type4.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,
4,5,MHS053,js02,type4,MHS053_js02_p03.py,MHS053_js02_p03_type4.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,


Diproses pada 2026-06-17 02:59:41


In [10]:
# ==============================================================================
# GENERATE TYPE 1 - TYPE 3 (TUGAS)
# ==============================================================================
print(f"{'GENERATE KLON TUGAS (TYPE 1 - 4)':^50}")
EVAL_TYPE1_DIR = os.path.join(EVAL_T["SAMPLE"], "type1")
EVAL_TYPE2_DIR = os.path.join(EVAL_T["SAMPLE"], "type2")
EVAL_TYPE3_DIR = os.path.join(EVAL_T["SAMPLE"], "type3")
EVAL_TYPE4_DIR = os.path.join(EVAL_T["SAMPLE"], "type4")
REPORT_PATH = RESULTS_EVAL_T["SAMPLING_REPORT"]

df_type1 = generate_clone_dataset(
    clone_type="type1",
    sample_report_path=REPORT_PATH,
    target_dir=EVAL_TYPE1_DIR
)

df_type2 = generate_clone_dataset(
    clone_type="type2",
    sample_report_path=REPORT_PATH,
    target_dir=EVAL_TYPE2_DIR
)

df_type3 = generate_clone_dataset(
    clone_type="type3",
    sample_report_path=REPORT_PATH,
    target_dir=EVAL_TYPE3_DIR
)

df_type4 = generate_clone_dataset(
    clone_type="type4",
    sample_report_path=REPORT_PATH,
    target_dir=EVAL_TYPE4_DIR
)

export_clone_log(
    df_type1,
    REPORT_PATH,
    "Type1_Code"
)

export_clone_log(
    df_type2,
    REPORT_PATH,
    "Type2_Code"
)

export_clone_log(
    df_type3,
    REPORT_PATH,
    "Type3_Code"
)

export_clone_log(
    df_type4,
    REPORT_PATH,
    "Type4_Code"
)

display(df_type1.head())
display(df_type2.head())
display(df_type3.head())
display(df_type4.head())
print(f"Diproses pada {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

         GENERATE KLON TUGAS (TYPE 1 - 4)         
GENERATE TYPE1
Total File  : 170
Success     : 170
Failed      : 0
Output      : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\tugas\06_SAMPLE\type1
GENERATE TYPE2
Total File  : 170
Success     : 170
Failed      : 0
Output      : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\tugas\06_SAMPLE\type2
GENERATE TYPE3
Total File  : 170
Success     : 170
Failed      : 0
Output      : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\tugas\06_SAMPLE\type3
GENERATE TYPE4
Total File  : 170
Success     : 170
Failed      : 0
Output      : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)_eval\tugas\06_SAMPLE\type4


,No,NIM,Modul,Clone_Type,Nama_File_Asli,Nama_File_Clone,Path_Asli,Path_Clone,Status,Keterangan
0,1,MHS004,js02,type1,MHS004_js02_tp.py,MHS004_js02_tp_type1.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,
1,2,MHS018,js02,type1,MHS018_js02_tp.py,MHS018_js02_tp_type1.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,
2,3,MHS006,js02,type1,MHS006_js02_tp.py,MHS006_js02_tp_type1.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,
3,4,MHS029,js02,type1,MHS029_js02_tp.py,MHS029_js02_tp_type1.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,
4,5,MHS052,js02,type1,MHS052_js02_tp.py,MHS052_js02_tp_type1.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,


,No,NIM,Modul,Clone_Type,Nama_File_Asli,Nama_File_Clone,Path_Asli,Path_Clone,Status,Keterangan
0,1,MHS004,js02,type2,MHS004_js02_tp.py,MHS004_js02_tp_type2.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,
1,2,MHS018,js02,type2,MHS018_js02_tp.py,MHS018_js02_tp_type2.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,
2,3,MHS006,js02,type2,MHS006_js02_tp.py,MHS006_js02_tp_type2.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,
3,4,MHS029,js02,type2,MHS029_js02_tp.py,MHS029_js02_tp_type2.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,
4,5,MHS052,js02,type2,MHS052_js02_tp.py,MHS052_js02_tp_type2.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,


,No,NIM,Modul,Clone_Type,Nama_File_Asli,Nama_File_Clone,Path_Asli,Path_Clone,Status,Keterangan
0,1,MHS004,js02,type3,MHS004_js02_tp.py,MHS004_js02_tp_type3.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,
1,2,MHS018,js02,type3,MHS018_js02_tp.py,MHS018_js02_tp_type3.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,
2,3,MHS006,js02,type3,MHS006_js02_tp.py,MHS006_js02_tp_type3.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,
3,4,MHS029,js02,type3,MHS029_js02_tp.py,MHS029_js02_tp_type3.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,
4,5,MHS052,js02,type3,MHS052_js02_tp.py,MHS052_js02_tp_type3.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,


,No,NIM,Modul,Clone_Type,Nama_File_Asli,Nama_File_Clone,Path_Asli,Path_Clone,Status,Keterangan
0,1,MHS004,js02,type4,MHS004_js02_tp.py,MHS004_js02_tp_type4.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,
1,2,MHS018,js02,type4,MHS018_js02_tp.py,MHS018_js02_tp_type4.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,
2,3,MHS006,js02,type4,MHS006_js02_tp.py,MHS006_js02_tp_type4.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,
3,4,MHS029,js02,type4,MHS029_js02_tp.py,MHS029_js02_tp_type4.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,
4,5,MHS052,js02,type4,MHS052_js02_tp.py,MHS052_js02_tp_type4.py,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\datas...,SUCCESS,


Diproses pada 2026-06-17 03:00:44
